# Магистерский эксперимент v11

## Послойное и вычислительно сопоставимое сравнение BP, Exact, FixedPred и Strict в LeNet-подобных сверточных сетях

Ноутбук разделяет обязательное ядро ВКР и публикационные расширения. Все дорогие блоки управляются флагами и используют только заранее выбранные конфигурации.

## 0. Порядок работы

1. Создать окружение по `README_RUN_RESEARCH_v11_keyboard_safe.md`.
2. Выполнить `python preflight_check_v11.py`.
3. Запустить режим `smoke`, сохранить полный хеш Torch2PC и точный lock-файл.
4. Выполнить H0/H1 на CPU и, при наличии, на GPU.
5. Запустить pilot минимум на трех seed.
6. Зафиксировать выбранные конфигурации до открытия test.
7. Выполнить основное сравнение на десяти парных model_seed.
8. Запустить послойный, compute-matched и robustness анализы.
9. Выполнить абляции optimizer/split в публикационном режиме.
10. Экспортировать таблицы, манифест и публикационный пакет.

In [ ]:
# 1. Импорты
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import platform
import random
import re
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy import stats
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# 2. Конфигурация

@dataclass
class ExperimentConfig:
    run_mode: str = "smoke"  # smoke, pilot, final
    analysis_tier: str = "core"  # core, extended, publication

    # Независимые источники случайности
    seed: int = 42  # технический запуск
    split_seed: int = 42
    loader_seed: int = 31415
    corruption_seed: int = 20250325
    pilot_seeds: list[int] = field(default_factory=lambda: [40, 41, 42])
    final_seeds: list[int] = field(default_factory=lambda: list(range(10)))
    extended_final_seeds: list[int] = field(default_factory=lambda: list(range(5)))

    # Правила статистического анализа
    minimum_final_seeds: int = 10
    wilcoxon_min_pairs: int = 5
    confirmatory_min_pairs: int = 10
    stability_min_seeds: int = 5
    pilot_min_success_rate: float = 2.0 / 3.0

    # Публикационные диагностические бюджеты
    gradient_batches: int = 5
    representation_max_samples: int = 1000
    representation_sample_seed: int = 20251001
    representation_bootstrap_repeats: int = 200
    representation_bootstrap_sample_size: int = 256
    timing_batch_sizes: list[int] = field(default_factory=lambda: [64, 128])
    compute_matched_seeds: list[int] = field(default_factory=lambda: list(range(5)))
    split_sensitivity_seeds: list[int] = field(default_factory=lambda: [42, 142, 242])
    split_sensitivity_model_seeds: list[int] = field(default_factory=lambda: [0, 1, 2])
    optimizer_ablation_names: list[str] = field(default_factory=lambda: ["Adam", "SGD"])
    optimizer_ablation_seeds: list[int] = field(default_factory=lambda: list(range(5)))

    datasets_required: list[str] = field(default_factory=lambda: ["MNIST", "FashionMNIST"])
    datasets_optional: list[str] = field(default_factory=lambda: ["KMNIST"])
    publication_datasets: list[str] = field(default_factory=lambda: ["MNIST", "FashionMNIST", "KMNIST"])
    h4_source_dataset: str = "MNIST"
    h4_target_dataset: str = "FashionMNIST"

    validation_fraction: float = 0.15
    batch_size: int = 128
    num_workers: int = 2
    learning_rate: float = 1e-3
    optimizer_name: str = "Adam"
    sgd_learning_rate: float = 1e-2
    pilot_epochs: int = 2
    final_epochs: int = 10
    early_stopping_patience: int = 3
    primary_metric: str = "macro_f1"

    torch2pc_repo: str = "https://github.com/RobertRosenbaum/Torch2PC.git"
    torch2pc_commit: str = ""
    rosenbaum_correction_doi: str = "10.1371/journal.pone.0320944"
    require_correction_audit: bool = True

    trajectory_n_values: list[int] = field(default_factory=lambda: [1, 2, 5, 6, 10, 20, 50])
    trajectory_max_batches: int = 1
    trajectory_max_samples: int = 128

    core_corruptions: list[str] = field(default_factory=lambda: [
        "gaussian_noise", "gaussian_blur", "occlusion"
    ])
    extended_corruptions: list[str] = field(default_factory=lambda: [
        "gaussian_noise", "salt_pepper", "occlusion", "rotation",
        "gaussian_blur", "low_contrast", "pixel_dropout"
    ])
    diagnostic_corruptions: list[str] = field(default_factory=lambda: [
        "gaussian_noise", "gaussian_blur", "occlusion"
    ])
    diagnostic_severities: list[int] = field(default_factory=lambda: [1, 3, 5])

    literature_corpus_file: str = "uploaded_pdf_corpus_v10_keyboard_safe.md"
    patent_search_file: str = "patent_search_protocol_v10_keyboard_safe.md"

    fixedpred_grid: list[dict[str, float | int]] = field(default_factory=lambda: [
        {"eta": 0.1, "n": 1},
        {"eta": 0.1, "n": 5},
        {"eta": 0.1, "n": 10},
        {"eta": 0.3, "n": 1},
        {"eta": 0.3, "n": 5},
        {"eta": 0.3, "n": 10},
        {"eta": 0.5, "n": 5},
        {"eta": 0.5, "n": 10},
        {"eta": 1.0, "n": 6},
    ])
    strict_grid: list[dict[str, float | int]] = field(default_factory=lambda: [
        {"eta": 0.01, "n": 5},
        {"eta": 0.01, "n": 20},
        {"eta": 0.05, "n": 10},
        {"eta": 0.05, "n": 20},
        {"eta": 0.1, "n": 10},
        {"eta": 0.1, "n": 20},
    ])

    exact_cosine_min: float = 0.9999
    exact_relative_l2_max: float = 1e-4
    fixedpred_cosine_min: float = 0.9999
    fixedpred_relative_l2_max: float = 1e-4

CONFIG_PATH = Path(os.environ.get("EXPERIMENT_CONFIG_PATH", "configs/experiment_config_ubuntu_v11.json"))
if CONFIG_PATH.exists():
    raw = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    known = {k: v for k, v in raw.items() if k in ExperimentConfig.__dataclass_fields__}
    if "sanity_thresholds" in raw:
        known["exact_cosine_min"] = raw["sanity_thresholds"].get("exact_cosine_min", 0.9999)
        known["exact_relative_l2_max"] = raw["sanity_thresholds"].get("exact_relative_l2_max", 1e-4)
        known["fixedpred_cosine_min"] = raw["sanity_thresholds"].get("fixedpred_cosine_min", 0.9999)
        known["fixedpred_relative_l2_max"] = raw["sanity_thresholds"].get("fixedpred_relative_l2_max", 1e-4)
    CFG = ExperimentConfig(**known)
else:
    CFG = ExperimentConfig()

if CFG.analysis_tier not in {"core", "extended", "publication"}:
    raise ValueError("analysis_tier must be core, extended, or publication")
if len(CFG.final_seeds) < CFG.minimum_final_seeds:
    raise ValueError(
        f"final_seeds must contain at least {CFG.minimum_final_seeds} values"
    )

def active_dataset_names() -> list[str]:
    return (
        list(CFG.publication_datasets)
        if CFG.analysis_tier == "publication"
        else list(CFG.datasets_required)
    )


_RUN_STAGE = os.environ.get("RUN_STAGE", "smoke").strip().lower()
_VALID_STAGES = {
    "smoke", "pilot", "final", "dataset-transfer", "layerwise",
    "robustness", "architecture", "trajectory", "benchmark",
    "publication", "all-diagnostics", "export"
}
if _RUN_STAGE not in _VALID_STAGES:
    raise ValueError(f"Unknown RUN_STAGE={_RUN_STAGE!r}")

RUN_PILOT_SEARCH = _RUN_STAGE == "pilot"
RUN_FINAL_EXPERIMENTS = _RUN_STAGE == "final"
RUN_DATASET_TRANSFER_ANALYSIS = _RUN_STAGE in {"dataset-transfer", "all-diagnostics", "publication"}
RUN_LAYERWISE_ANALYSIS = _RUN_STAGE in {"layerwise", "all-diagnostics", "publication"}
RUN_ROBUSTNESS_ANALYSIS = _RUN_STAGE in {"robustness", "all-diagnostics", "publication"}
RUN_ARCHITECTURE_TRANSFER_ANALYSIS = _RUN_STAGE in {"architecture", "all-diagnostics", "publication"}
RUN_INFERENCE_TRAJECTORY = _RUN_STAGE in {"trajectory", "all-diagnostics", "publication"}
RUN_STABILITY_ANALYSIS = _RUN_STAGE in {"trajectory", "all-diagnostics", "publication"}
RUN_COMPUTE_MATCHED_ANALYSIS = _RUN_STAGE in {"benchmark", "publication"}
RUN_OPTIMIZER_ABLATION = _RUN_STAGE == "publication"
RUN_SPLIT_SENSITIVITY = _RUN_STAGE == "publication"
RUN_REPRESENTATION_BOOTSTRAP = _RUN_STAGE in {"layerwise", "all-diagnostics", "publication"}

_FORCE_DEVICE = os.environ.get("FORCE_DEVICE", "auto").strip().lower()
if _FORCE_DEVICE == "cpu":
    DEVICE = torch.device("cpu")
elif _FORCE_DEVICE in {"gpu", "cuda", "rocm"}:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU was requested, but torch.cuda.is_available() is False")
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULTS_DIR = Path("results")
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CONFIG_DIR = RESULTS_DIR / "config"
FIGURE_DIR = RESULTS_DIR / "figures"
LOG_DIR = RESULTS_DIR / "logs"
METRIC_DIR = RESULTS_DIR / "metrics"
SPLIT_DIR = RESULTS_DIR / "splits"
for directory in [CHECKPOINT_DIR, CONFIG_DIR, FIGURE_DIR, LOG_DIR, METRIC_DIR, SPLIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(asdict(CFG))
print("Device:", DEVICE)

In [ ]:
# 3. Воспроизводимость и снимок окружения

def stable_int_seed(*parts: Any) -> int:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return int.from_bytes(digest[:8], "big") % (2**31 - 1)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception:
            pass


def git_head(path: Path) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            text=True,
        ).strip()
    except Exception:
        return "unavailable"


def capture_requirements_lock(
    output_path: Path = CONFIG_DIR / "requirements-lock-v11.txt",
) -> Path:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        check=True,
        capture_output=True,
        text=True,
    )
    output_path.write_text(result.stdout, encoding="utf-8")
    return output_path


def environment_snapshot(torch2pc_path: Optional[Path] = None) -> dict[str, Any]:
    snapshot = {
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": __import__("torchvision").__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "device": str(DEVICE),
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "hip_version": getattr(torch.version, "hip", None),
        "cudnn_version": torch.backends.cudnn.version(),
        "config": asdict(CFG),
    }
    if torch2pc_path is not None:
        snapshot["torch2pc_commit"] = git_head(torch2pc_path)
    return snapshot

set_seed(CFG.seed)
_requested_threads = int(os.environ.get("TORCH_NUM_THREADS", "0"))
if _requested_threads > 0:
    torch.set_num_threads(_requested_threads)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass
(CONFIG_DIR / "resolved_config.json").write_text(
    json.dumps(asdict(CFG), ensure_ascii=False, indent=2),
    encoding="utf-8",
)
lock_path = capture_requirements_lock()
print("Saved exact environment:", lock_path)

## 1. Torch2PC и поправка Rosenbaum 2025

Официальный Torch2PC содержит функцию `PCInfer` и режимы `Strict`, `FixedPred`, `Exact`.

Основная статья Rosenbaum 2022 используется совместно с официальной поправкой 2025:

- Rosenbaum, R. Correction: On the relationship between predictive coding and backpropagation. PLOS ONE, 2025, 20(3), e0320944.
- DOI: `10.1371/journal.pone.0320944`.

Перед H0/H1 ноутбук проверяет ключевые признаки исправленной реализации:

1. Strict пересчитывает ошибки на каждой итерации.
2. Ошибка Strict имеет вид prediction - belief.
3. Обновление состояния использует `epsilon - VJP`.
4. Ошибка FixedPred имеет вид feedforward activation - belief.
5. Exact использует `belief = activation - epsilon`.

Аудит является проверкой реализации, а не заменой математического доказательства. Для pilot/final требуется полный зафиксированный хеш коммита Torch2PC.

In [ ]:
# 4. Клонирование и импорт Torch2PC

if CFG.run_mode in {"pilot", "final"} and not re.fullmatch(r"[0-9a-fA-F]{40}", CFG.torch2pc_commit):
    raise RuntimeError(
        "For pilot/final runs, torch2pc_commit must contain the full 40-character commit SHA detected during smoke run."
    )

EXTERNAL_DIR = Path("external")
TORCH2PC_DIR = EXTERNAL_DIR / "Torch2PC"
EXTERNAL_DIR.mkdir(exist_ok=True)

if not TORCH2PC_DIR.exists():
    subprocess.run(
        ["git", "clone", CFG.torch2pc_repo, str(TORCH2PC_DIR)],
        check=True,
    )

if CFG.torch2pc_commit:
    subprocess.run(["git", "-C", str(TORCH2PC_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(TORCH2PC_DIR), "checkout", CFG.torch2pc_commit], check=True)

if str(TORCH2PC_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(TORCH2PC_DIR.resolve()))

import TorchSeq2PC
from TorchSeq2PC import PCInfer

TORCH2PC_HEAD = git_head(TORCH2PC_DIR)
print("Torch2PC commit:", TORCH2PC_HEAD)
(CONFIG_DIR / "detected_torch2pc_commit.txt").write_text(TORCH2PC_HEAD + "\n", encoding="utf-8")

if CFG.torch2pc_commit and TORCH2PC_HEAD != CFG.torch2pc_commit:
    raise RuntimeError(
        f"Torch2PC checkout mismatch: expected {CFG.torch2pc_commit}, got {TORCH2PC_HEAD}"
    )

snapshot = environment_snapshot(TORCH2PC_DIR)
(LOG_DIR / "environment_snapshot.json").write_text(
    json.dumps(snapshot, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

## 1A. Аудит соответствия поправке Rosenbaum 2025

Этот блок проверяет исходный код зафиксированной версии Torch2PC по ключевым исправлениям, перечисленным в официальной поправке. Результат сохраняется до запуска контрольных экспериментов.

In [ ]:
# 4A. Аудит исходного кода Torch2PC по поправке Rosenbaum 2025

def _function_source(source_text: str, function_name: str) -> str:
    pattern = re.compile(
        rf"^def\s+{re.escape(function_name)}\s*\(.*?(?=^def\s+|\Z)",
        flags=re.MULTILINE | re.DOTALL,
    )
    match = pattern.search(source_text)
    if match is None:
        raise RuntimeError(f"Function {function_name} not found in TorchSeq2PC.py")
    return match.group(0)


def audit_rosenbaum_2025(torch2pc_file: Path) -> dict[str, Any]:
    source = torch2pc_file.read_text(encoding="utf-8")
    strict_src = _function_source(source, "StrictPCPredErrs")
    fixed_src = _function_source(source, "FixedPredPCPredErrs")
    exact_src = _function_source(source, "ExactPredErrs")

    def compact(value: str) -> str:
        return re.sub(r"\s+", "", value)

    strict_compact = compact(strict_src)
    fixed_compact = compact(fixed_src)
    exact_compact = compact(exact_src)

    strict_loop_pos = strict_compact.find("foriinrange(n):")
    strict_error_pos = strict_compact.find("epsilon[layer]=model[layer-1](v[layer-1])-v[layer]")

    checks = {
        "strict_recomputes_errors_each_iteration": strict_loop_pos >= 0 and strict_error_pos > strict_loop_pos,
        "strict_error_is_prediction_minus_belief": "epsilon[layer]=model[layer-1](v[layer-1])-v[layer]" in strict_compact,
        "strict_update_is_epsilon_minus_vjp": "dv=epsilon[layer]-epsdfdv" in strict_compact,
        "fixedpred_error_is_activation_minus_belief": "epsilon[layer]=vhat[layer]-v[layer]" in fixed_compact,
        "fixedpred_update_is_epsilon_minus_vjp": "dv=epsilon[layer]-epsdfdv" in fixed_compact,
        "exact_belief_is_activation_minus_epsilon": "v[layer]=vhat[layer]-epsilon[layer]" in exact_compact,
    }
    result = {
        "source": "Rosenbaum 2025 Correction",
        "doi": CFG.rosenbaum_correction_doi,
        "torch2pc_commit": TORCH2PC_HEAD,
        "torch2pc_file": str(torch2pc_file),
        "checks": checks,
        "passed": bool(all(checks.values())),
    }
    return result


CORRECTION_AUDIT = audit_rosenbaum_2025(TORCH2PC_DIR / "TorchSeq2PC.py")
(CONFIG_DIR / "rosenbaum_2025_correction_audit.json").write_text(
    json.dumps(CORRECTION_AUDIT, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(json.dumps(CORRECTION_AUDIT, ensure_ascii=False, indent=2))

if CFG.require_correction_audit and not CORRECTION_AUDIT["passed"]:
    raise RuntimeError(
        "Torch2PC source audit does not match the Rosenbaum 2025 correction. "
        "Do not continue until the implementation and pinned commit are reviewed."
    )

## 1B. Трассировка литературы в экспериментальный протокол

Этот раздел сохраняет явную связь между загруженными публикациями и решениями методологии. Он не предполагает, что разные архитектуры PC эквивалентны Torch2PC.

In [ ]:
# 4B. Карта "источник -> решение эксперимента"

LITERATURE_DESIGN_MAP = {
    "WhittingtonBogacz2017": {
        "local_file": "emss-73010.pdf",
        "role": "Связь локального PC-обучения и BP",
        "design": ["H0-H3", "layerwise_gradient_similarity"],
    },
    "Millidge2020": {
        "local_file": "2006.04182v5.pdf",
        "role": "PC на CNN и вычислительных графах",
        "design": ["novelty_boundary", "pc_step_runtime"],
    },
    "PCX2025": {
        "local_file": "9795_Benchmarking_Predictive_C.pdf",
        "role": "Единый benchmark и ограничения масштабирования",
        "design": ["reproducibility", "failure_catalog", "runtime_budget"],
    },
    "Alamia2021": {
        "local_file": "2106.04225v1.pdf",
        "role": "Обратная связь при шуме",
        "design": ["gaussian_noise", "layer_error_trends"],
    },
    "Han2018": {
        "local_file": "1805.07526v2.pdf",
        "role": "Динамика представлений по итерациям",
        "design": ["pcinfer_trajectory"],
    },
    "Boutin2021": {
        "local_file": "journal.pcbi.1008629.pdf",
        "role": "Послойные представления, шум и размытие",
        "design": ["gaussian_blur", "layer_error_trends"],
    },
    "Spratling2017": {
        "local_file": "s12559-016-9445-1.pdf",
        "role": "Устойчивость к перекрытию",
        "design": ["occlusion"],
    },
    "Kirubeswaran2023": {
        "local_file": "2301.07455v2.pdf",
        "role": "Вариативность между экземплярами",
        "design": ["seed_stability", "failure_catalog"],
    },
    "Rane2020": {
        "local_file": "1906.11902v3.pdf",
        "role": "Критический аудит архитектур PC",
        "design": ["implementation_audit", "terminology_boundary"],
    },
    "CPC_boundary": {
        "local_files": ["1807.03748v2.pdf", "1905.09272v3.pdf"],
        "role": "Отдельное направление Contrastive Predictive Coding",
        "design": ["terminology_only"],
    },
}

(CONFIG_DIR / "literature_design_traceability.json").write_text(
    json.dumps(LITERATURE_DESIGN_MAP, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Literature traceability entries:", len(LITERATURE_DESIGN_MAP))

## 2. Данные: train, validation, test

Поисковые функции ниже не принимают test loader. Это структурно снижает риск случайного выбора гиперпараметров по тестовой выборке.

In [ ]:
# 5. Подготовка данных и сохранение split indices

def dataset_class(name: str):
    mapping = {
        "MNIST": datasets.MNIST,
        "FashionMNIST": datasets.FashionMNIST,
        "KMNIST": datasets.KMNIST,
        "EMNIST": datasets.EMNIST,
    }
    if name not in mapping:
        raise ValueError(f"Unknown dataset: {name}")
    return mapping[name]


def image_transform():
    return transforms.Compose([transforms.Pad(2), transforms.ToTensor()])


def dataset_targets(dataset) -> np.ndarray:
    targets = getattr(dataset, "targets", None)
    if targets is None:
        raise AttributeError("Dataset does not expose targets")
    if torch.is_tensor(targets):
        targets = targets.cpu().numpy()
    return np.asarray(targets)


def stratified_subset_indices(
    targets: np.ndarray,
    size: Optional[int],
    split_seed: int,
) -> np.ndarray:
    all_indices = np.arange(len(targets))
    if size is None or size >= len(all_indices):
        return all_indices
    selected, _ = train_test_split(
        all_indices,
        train_size=size,
        random_state=split_seed,
        stratify=targets,
    )
    return np.sort(selected)


def split_file_path(
    dataset_name: str,
    split_seed: int,
    train_total_subset: Optional[int],
    test_subset: Optional[int],
) -> Path:
    train_tag = "full" if train_total_subset is None else str(train_total_subset)
    test_tag = "full" if test_subset is None else str(test_subset)
    return SPLIT_DIR / (
        f"{dataset_name}_splitseed{split_seed}_train{train_tag}_test{test_tag}_indices.npz"
    )


def save_split(
    dataset_name: str,
    split_seed: int,
    train_total_subset: Optional[int],
    test_subset: Optional[int],
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    test_idx: np.ndarray,
) -> Path:
    path = split_file_path(
        dataset_name, split_seed, train_total_subset, test_subset
    )
    np.savez_compressed(
        path,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        split_seed=np.asarray([split_seed]),
    )
    return path


def build_dataloaders(
    name: str,
    split_seed: Optional[int] = None,
    loader_seed: Optional[int] = None,
    batch_size: Optional[int] = None,
    validation_fraction: Optional[float] = None,
    train_total_subset: Optional[int] = None,
    test_subset: Optional[int] = None,
    include_test: bool = True,
) -> dict[str, Any]:
    split_seed = CFG.split_seed if split_seed is None else int(split_seed)
    loader_seed = CFG.seed if loader_seed is None else int(loader_seed)
    batch_size = batch_size or CFG.batch_size
    validation_fraction = (
        CFG.validation_fraction
        if validation_fraction is None
        else validation_fraction
    )
    cls = dataset_class(name)
    kwargs = {"split": "digits"} if name == "EMNIST" else {}

    full_train = cls(
        root="data", train=True, download=True,
        transform=image_transform(), **kwargs
    )
    full_test = (
        cls(
            root="data",
            train=False,
            download=True,
            transform=image_transform(),
            **kwargs,
        )
        if include_test
        else None
    )

    train_targets = dataset_targets(full_train)
    test_targets = (
        dataset_targets(full_test)
        if full_test is not None
        else np.asarray([], dtype=int)
    )

    selected_train = stratified_subset_indices(
        train_targets, train_total_subset, split_seed
    )
    train_idx, val_idx = train_test_split(
        selected_train,
        test_size=validation_fraction,
        random_state=split_seed,
        stratify=train_targets[selected_train],
    )
    train_idx = np.sort(train_idx)
    val_idx = np.sort(val_idx)
    test_idx = (
        stratified_subset_indices(
            test_targets,
            test_subset,
            split_seed + 1000,
        )
        if include_test
        else np.asarray([], dtype=int)
    )

    split_path = save_split(
        name,
        split_seed,
        train_total_subset,
        test_subset,
        train_idx,
        val_idx,
        test_idx,
    )

    generator = torch.Generator()
    generator.manual_seed(loader_seed)
    common = {
        "batch_size": batch_size,
        "num_workers": CFG.num_workers,
        "pin_memory": DEVICE.type == "cuda",
        "worker_init_fn": seed_worker,
        "persistent_workers": CFG.num_workers > 0,
    }

    loaders: dict[str, Any] = {
        "train": DataLoader(
            Subset(full_train, train_idx.tolist()),
            shuffle=True,
            generator=generator,
            **common,
        ),
        "validation": DataLoader(
            Subset(full_train, val_idx.tolist()),
            shuffle=False,
            **common,
        ),
        "split_path": split_path,
    }
    if include_test:
        if full_test is None:
            raise RuntimeError("Test dataset was not created")
        loaders["test"] = DataLoader(
            Subset(full_test, test_idx.tolist()),
            shuffle=False,
            **common,
        )
    return loaders


def mode_subset_sizes() -> tuple[Optional[int], Optional[int]]:
    if CFG.run_mode == "smoke":
        return 1200, 400
    if CFG.run_mode == "pilot":
        return 12000, 2000
    return None, None

train_subset, test_subset = mode_subset_sizes()
loaders = build_dataloaders(
    CFG.datasets_required[0],
    split_seed=CFG.split_seed,
    loader_seed=CFG.loader_seed,
    train_total_subset=train_subset,
    test_subset=test_subset,
)
for split_name in ["train", "validation", "test"]:
    loader = loaders[split_name]
    print(split_name, len(loader.dataset), next(iter(loader))[0].shape)
print("Split file:", loaders["split_path"])

## 3. Модели

Верхний `nn.Sequential` определяет PC-слои. Внутренние блоки объединяют свертку, активацию и pooling в один слой Torch2PC.

In [ ]:
# 6. Архитектуры LeNet

class Flatten(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.flatten(x, 1)


def make_lenet_classic(num_classes: int = 10) -> nn.Sequential:
    return nn.Sequential(
        nn.Sequential(nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2, 2)),
        nn.Sequential(nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2, 2)),
        Flatten(),
        nn.Sequential(nn.Linear(16 * 5 * 5, 120), nn.Tanh()),
        nn.Sequential(nn.Linear(120, 84), nn.Tanh()),
        nn.Linear(84, num_classes),
    )


def make_lenet_modern(num_classes: int = 10) -> nn.Sequential:
    return nn.Sequential(
        nn.Sequential(nn.Conv2d(1, 6, 5), nn.ReLU(), nn.MaxPool2d(2, 2)),
        nn.Sequential(nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2, 2)),
        Flatten(),
        nn.Sequential(nn.Linear(16 * 5 * 5, 120), nn.ReLU()),
        nn.Sequential(nn.Linear(120, 84), nn.ReLU()),
        nn.Linear(84, num_classes),
    )


def make_lenet_small(num_classes: int = 10) -> nn.Sequential:
    return nn.Sequential(
        nn.Sequential(nn.Conv2d(1, 4, 5), nn.Tanh(), nn.AvgPool2d(2, 2)),
        nn.Sequential(nn.Conv2d(4, 8, 5), nn.Tanh(), nn.AvgPool2d(2, 2)),
        Flatten(),
        nn.Sequential(nn.Linear(8 * 5 * 5, 64), nn.Tanh()),
        nn.Linear(64, num_classes),
    )

MODEL_FACTORIES: dict[str, Callable[[], nn.Sequential]] = {
    "lenet_classic": make_lenet_classic,
    "lenet_modern": make_lenet_modern,
    "lenet_small": make_lenet_small,
}


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for name, factory in MODEL_FACTORIES.items():
    model = factory().to(DEVICE)
    xb, _ = next(iter(loaders["train"]))
    with torch.no_grad():
        logits = model(xb[:4].to(DEVICE))
    print(name, "layers=", len(model), "params=", count_parameters(model), "output=", tuple(logits.shape))

## 4. Метрики, оценка и контрольные точки

In [ ]:
# 7. Метрики классификации

def ensure_finite_tensor(name: str, value: torch.Tensor) -> None:
    if not torch.isfinite(value).all():
        raise FloatingPointError(f"Non-finite values detected in {name}")


def gradient_diagnostics(model: nn.Module) -> dict[str, float]:
    squared_norm = 0.0
    max_abs = 0.0
    n_tensors = 0
    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            continue
        ensure_finite_tensor(f"gradient:{name}", parameter.grad)
        grad = parameter.grad.detach().float()
        squared_norm += float(torch.sum(grad * grad).item())
        max_abs = max(max_abs, float(torch.max(torch.abs(grad)).item()))
        n_tensors += 1
    if n_tensors == 0:
        raise RuntimeError("No parameter gradients were produced")
    return {
        "gradient_l2": math.sqrt(squared_norm),
        "gradient_max_abs": max_abs,
        "gradient_tensors": n_tensors,
    }


def ensure_finite_parameters(model: nn.Module) -> None:
    for name, parameter in model.named_parameters():
        ensure_finite_tensor(f"parameter:{name}", parameter.detach())


def evaluate_classifier(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device = DEVICE,
) -> dict[str, Any]:
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss = 0.0
    y_true: list[int] = []
    y_pred: list[int] = []
    probabilities: list[np.ndarray] = []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            ensure_finite_tensor("evaluation_logits", logits)
            batch_loss = loss_fn(logits, yb)
            ensure_finite_tensor("evaluation_loss", batch_loss)
            total_loss += float(batch_loss.item())
            prob = torch.softmax(logits, dim=1)
            ensure_finite_tensor("evaluation_probabilities", prob)
            pred = prob.argmax(dim=1)
            y_true.extend(yb.cpu().tolist())
            y_pred.extend(pred.cpu().tolist())
            probabilities.append(prob.cpu().numpy())

    n = len(y_true)
    if n == 0:
        raise RuntimeError("Evaluation loader produced no samples")
    return {
        "loss": total_loss / n,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "probabilities": np.concatenate(probabilities, axis=0),
    }


def metric_row(metrics: dict[str, Any]) -> dict[str, float]:
    return {k: float(metrics[k]) for k in ["loss", "accuracy", "macro_f1"]}

In [ ]:
# 8. Обучение с выбором best-validation checkpoint

def sync_device() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def make_optimizer(
    model: nn.Module,
    optimizer_name: str,
    learning_rate: float,
) -> torch.optim.Optimizer:
    name = optimizer_name.strip().lower()
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=learning_rate)
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    raise ValueError(f"Unknown optimizer: {optimizer_name}")


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    method: str,
    eta: Optional[float] = None,
    n: Optional[int] = None,
    max_batches: Optional[int] = None,
) -> dict[str, float]:
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_samples = 0
    max_gradient_l2 = 0.0
    max_gradient_abs = 0.0
    start = time.perf_counter()

    for batch_idx, (xb, yb) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        if method == "BP":
            logits = model(xb)
            ensure_finite_tensor("training_logits", logits)
            loss = loss_fn(logits, yb)
            ensure_finite_tensor("training_loss", loss)
            loss.backward()
        elif method in {"Exact", "FixedPred", "Strict"}:
            _, loss, _, _, _ = PCInfer(
                model,
                loss_fn,
                xb,
                yb,
                method,
                eta=0.1 if eta is None else float(eta),
                n=20 if n is None else int(n),
            )
            ensure_finite_tensor("pc_training_loss", loss)
        else:
            raise ValueError(f"Unknown method: {method}")

        grad_info = gradient_diagnostics(model)
        max_gradient_l2 = max(max_gradient_l2, grad_info["gradient_l2"])
        max_gradient_abs = max(max_gradient_abs, grad_info["gradient_max_abs"])

        optimizer.step()
        ensure_finite_parameters(model)

        batch_size = int(yb.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size

    sync_device()
    elapsed = time.perf_counter() - start
    if total_samples == 0:
        raise RuntimeError("Training loader produced no samples")
    return {
        "train_loss": total_loss / total_samples,
        "epoch_time_sec": elapsed,
        "train_samples": total_samples,
        "max_gradient_l2": max_gradient_l2,
        "max_gradient_abs": max_gradient_abs,
    }


def fit_with_validation(
    model_factory: Callable[[], nn.Sequential],
    train_loader: DataLoader,
    validation_loader: DataLoader,
    method: str,
    seed: int,
    epochs: int,
    lr: float,
    eta: Optional[float] = None,
    n: Optional[int] = None,
    checkpoint_path: Optional[Path] = None,
    patience: Optional[int] = None,
    max_batches: Optional[int] = None,
    optimizer_name: Optional[str] = None,
    split_seed_value: Optional[int] = None,
    loader_seed_value: Optional[int] = None,
) -> tuple[nn.Module, pd.DataFrame, dict[str, Any]]:
    set_seed(seed)
    model = model_factory().to(DEVICE)
    optimizer_name = CFG.optimizer_name if optimizer_name is None else optimizer_name
    optimizer = make_optimizer(model, optimizer_name, lr)
    split_seed_value = CFG.split_seed if split_seed_value is None else int(split_seed_value)
    loader_seed_value = CFG.loader_seed if loader_seed_value is None else int(loader_seed_value)
    patience = CFG.early_stopping_patience if patience is None else patience

    best_metric = -np.inf
    best_epoch = -1
    best_state = None
    no_improvement = 0
    rows = []

    for epoch in range(1, epochs + 1):
        train_info = train_one_epoch(
            model,
            train_loader,
            optimizer,
            method=method,
            eta=eta,
            n=n,
            max_batches=max_batches,
        )
        val = evaluate_classifier(model, validation_loader)
        row = {
            "epoch": epoch,
            "method": method,
            "eta": eta,
            "n": n,
            **train_info,
            "val_loss": val["loss"],
            "val_accuracy": val["accuracy"],
            "val_macro_f1": val["macro_f1"],
        }
        row["cumulative_train_time_sec"] = float(
            sum(float(existing["epoch_time_sec"]) for existing in rows) + float(row["epoch_time_sec"])
        )
        rows.append(row)

        score = float(val[CFG.primary_metric])
        if not np.isfinite(score):
            raise FloatingPointError("Validation metric is not finite")
        if score > best_metric + 1e-12:
            best_metric = score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        if patience is not None and no_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError("No valid checkpoint was produced")
    model.load_state_dict(best_state)
    ensure_finite_parameters(model)

    metadata = {
        "method": method,
        "model_seed": seed,
        "split_seed": split_seed_value,
        "loader_seed": loader_seed_value,
        "optimizer": optimizer_name,
        "eta": eta,
        "n": n,
        "best_epoch": best_epoch,
        "best_validation_metric": best_metric,
        "torch2pc_commit": TORCH2PC_HEAD,
        "config": asdict(CFG),
    }
    if checkpoint_path is not None:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": best_state, "metadata": metadata}, checkpoint_path)

    return model, pd.DataFrame(rows), metadata


def load_checkpoint(
    model_factory: Callable[[], nn.Sequential],
    checkpoint_path: Path,
) -> tuple[nn.Module, dict[str, Any]]:
    payload = torch.load(checkpoint_path, map_location=DEVICE)
    model = model_factory().to(DEVICE)
    model.load_state_dict(payload["state_dict"])
    model.eval()
    ensure_finite_parameters(model)
    return model, payload.get("metadata", {})

## 5. H0 и H1: контроль эквивалентности

Сначала сравниваются градиенты на одинаковой инициализации и одном пакете.

- H0 проверяет Exact против обычного `Loss.backward()`.
- H1 проверяет теоретически точную конфигурацию FixedPred при `eta=1` и `n>=L`, где `L` - число верхнеуровневых элементов `nn.Sequential`, рассматриваемых Torch2PC как слои PC.

Аудит поправки Rosenbaum 2025, H0 и H1 образуют обязательный контрольный шлюз. Остальные результаты интерпретируются только при `passed: true` для всех трех проверок.

In [ ]:
# 9. Градиентные утилиты и H0

def named_gradient_vectors(model: nn.Module) -> dict[str, torch.Tensor]:
    result = {}
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        grad = torch.zeros_like(parameter) if parameter.grad is None else parameter.grad
        result[name] = grad.detach().flatten().cpu()
    return result


def cosine(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-12) -> float:
    return float(torch.dot(a, b) / (torch.norm(a) * torch.norm(b) + eps))


def relative_l2(reference: torch.Tensor, candidate: torch.Tensor, eps: float = 1e-12) -> float:
    return float(torch.norm(reference - candidate) / (torch.norm(reference) + eps))


def compare_gradient_maps(reference: dict[str, torch.Tensor], candidate: dict[str, torch.Tensor]) -> pd.DataFrame:
    rows = []
    for name in reference.keys() & candidate.keys():
        a, b = reference[name], candidate[name]
        rows.append({
            "parameter": name,
            "cosine": cosine(a, b),
            "relative_l2": relative_l2(a, b),
            "max_abs": float(torch.max(torch.abs(a - b))),
            "reference_norm": float(torch.norm(a)),
            "candidate_norm": float(torch.norm(b)),
        })
    return pd.DataFrame(rows)


def gradient_for_method(
    model: nn.Module,
    xb: torch.Tensor,
    yb: torch.Tensor,
    method: str,
    eta: Optional[float] = None,
    n: Optional[int] = None,
) -> tuple[dict[str, torch.Tensor], float]:
    loss_fn = nn.CrossEntropyLoss()
    model.zero_grad(set_to_none=True)
    if method == "BP":
        loss = loss_fn(model(xb), yb)
        loss.backward()
    else:
        _, loss, _, _, _ = PCInfer(
            model,
            loss_fn,
            xb,
            yb,
            method,
            eta=0.1 if eta is None else float(eta),
            n=20 if n is None else int(n),
        )
    return named_gradient_vectors(model), float(loss.item())


def run_exact_sanity(model_factory: Callable[[], nn.Sequential], loader: DataLoader, seed: int) -> tuple[pd.DataFrame, dict[str, Any]]:
    set_seed(seed)
    base = model_factory().to(DEVICE)
    bp_model = copy.deepcopy(base)
    exact_model = copy.deepcopy(base)
    xb, yb = next(iter(loader))
    model_dtype = next(base.parameters()).dtype
    xb, yb = xb.to(DEVICE, dtype=model_dtype), yb.to(DEVICE)

    bp_grad, bp_loss = gradient_for_method(bp_model, xb, yb, "BP")
    exact_grad, exact_loss = gradient_for_method(exact_model, xb, yb, "Exact")
    table = compare_gradient_maps(bp_grad, exact_grad)
    summary = {
        "bp_loss": bp_loss,
        "exact_loss": exact_loss,
        "min_cosine": float(table["cosine"].min()),
        "max_relative_l2": float(table["relative_l2"].max()),
        "max_abs": float(table["max_abs"].max()),
    }
    summary["passed"] = (
        summary["min_cosine"] >= CFG.exact_cosine_min
        and summary["max_relative_l2"] <= CFG.exact_relative_l2_max
    )
    return table, summary

exact_table, exact_summary = run_exact_sanity(make_lenet_classic, loaders["train"], CFG.seed)
display(exact_table)
print(exact_summary)
exact_table.to_csv(METRIC_DIR / "sanity_exact_layerwise.csv", index=False)
(CONFIG_DIR / "sanity_exact_summary.json").write_text(json.dumps(exact_summary, indent=2), encoding="utf-8")

In [ ]:
# 10. H1: FixedPred eta=1, n>=L

def run_fixedpred_control(model_factory: Callable[[], nn.Sequential], loader: DataLoader, seed: int) -> tuple[pd.DataFrame, dict[str, Any]]:
    set_seed(seed)
    base = model_factory().to(DEVICE)
    exact_model = copy.deepcopy(base)
    fixed_model = copy.deepcopy(base)
    xb, yb = next(iter(loader))
    model_dtype = next(base.parameters()).dtype
    xb, yb = xb.to(DEVICE, dtype=model_dtype), yb.to(DEVICE)
    n_layers = len(base)

    exact_grad, exact_loss = gradient_for_method(exact_model, xb, yb, "Exact")
    fixed_grad, fixed_loss = gradient_for_method(fixed_model, xb, yb, "FixedPred", eta=1.0, n=n_layers)
    table = compare_gradient_maps(exact_grad, fixed_grad)
    summary = {
        "exact_loss": exact_loss,
        "fixedpred_loss": fixed_loss,
        "eta": 1.0,
        "n": n_layers,
        "pc_depth": n_layers,
        "min_cosine": float(table["cosine"].min()),
        "max_relative_l2": float(table["relative_l2"].max()),
        "max_abs": float(table["max_abs"].max()),
    }
    summary["passed"] = (
        summary["min_cosine"] >= CFG.fixedpred_cosine_min
        and summary["max_relative_l2"] <= CFG.fixedpred_relative_l2_max
    )
    return table, summary

fixed_control_table, fixed_control_summary = run_fixedpred_control(make_lenet_classic, loaders["train"], CFG.seed)
display(fixed_control_table)
print(fixed_control_summary)
fixed_control_table.to_csv(METRIC_DIR / "sanity_fixedpred_control_layerwise.csv", index=False)
(CONFIG_DIR / "sanity_fixedpred_control_summary.json").write_text(
    json.dumps(fixed_control_summary, indent=2), encoding="utf-8"
)

control_gate_summary = {
    "rosenbaum_2025_correction_audit": bool(CORRECTION_AUDIT["passed"]),
    "H0_exact_vs_bp": bool(exact_summary["passed"]),
    "H1_fixedpred_exact_control": bool(fixed_control_summary["passed"]),
}
control_gate_summary["passed"] = bool(all(control_gate_summary.values()))
(CONFIG_DIR / "control_gate_summary.json").write_text(
    json.dumps(control_gate_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(control_gate_summary)

if not control_gate_summary["passed"]:
    raise RuntimeError(
        "Mandatory control gate failed: Rosenbaum 2025 audit, H0, and H1 must all pass "
        "before pilot or final PC experiments."
    )


## 5A. Контроль H0/H1 на доступных устройствах

CPU является обязательным контрольным устройством. При наличии CUDA те же проверки выполняются на GPU. Последующие эксперименты разрешены только после прохождения всех доступных контрольных устройств.

In [ ]:
# 10A. H0/H1 на CPU и GPU при наличии

def run_control_gate_on_device(device_name: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    global DEVICE
    previous_device = DEVICE
    previous_default_dtype = torch.get_default_dtype()
    DEVICE = torch.device(device_name)
    use_float64 = (
        device_name == "cpu"
        and os.environ.get("CORRECTNESS_FLOAT64", "1") == "1"
    )
    if use_float64:
        torch.set_default_dtype(torch.float64)
    try:
        exact_table_local, exact_summary_local = run_exact_sanity(
            make_lenet_classic, loaders["train"], CFG.seed
        )
        fixed_table_local, fixed_summary_local = run_fixedpred_control(
            make_lenet_classic, loaders["train"], CFG.seed
        )
        exact_table_local.insert(0, "device", device_name)
        fixed_table_local.insert(0, "device", device_name)
        summary = {
            "device": device_name,
            "H0_exact_vs_bp": bool(exact_summary_local["passed"]),
            "H1_fixedpred_exact_control": bool(fixed_summary_local["passed"]),
        }
        summary["passed"] = bool(summary["H0_exact_vs_bp"] and summary["H1_fixedpred_exact_control"])
        return pd.concat([
            exact_table_local.assign(control="H0"),
            fixed_table_local.assign(control="H1"),
        ], ignore_index=True), summary
    finally:
        DEVICE = previous_device
        torch.set_default_dtype(previous_default_dtype)

control_devices = ["cpu"]
if torch.cuda.is_available():
    control_devices.append("cuda")

control_device_tables = []
control_device_summaries = []
for device_name in control_devices:
    table, summary = run_control_gate_on_device(device_name)
    control_device_tables.append(table)
    control_device_summaries.append(summary)

control_by_device = pd.concat(control_device_tables, ignore_index=True)
control_by_device.to_csv(METRIC_DIR / "sanity_controls_by_device.csv", index=False)
(CONFIG_DIR / "sanity_controls_by_device.json").write_text(
    json.dumps(control_device_summaries, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
if not all(item["passed"] for item in control_device_summaries):
    raise RuntimeError("H0/H1 failed on at least one available device")
display(control_by_device)
print(control_device_summaries)

## 6. Пилотный поиск FixedPred и Strict

Функция принимает только train и validation loaders. Test loader здесь отсутствует намеренно.

In [ ]:
# 11. Поисковая сетка без test leakage

def safe_name(value: Any) -> str:
    return str(value).replace(".", "p").replace("/", "_")


def run_pc_search(
    dataset_name: str,
    model_name: str,
    method: str,
    grid: list[dict[str, float | int]],
    seed: int,
    epochs: int,
    train_total_subset: Optional[int],
    max_batches: Optional[int] = None,
) -> pd.DataFrame:
    if method not in {"FixedPred", "Strict"}:
        raise ValueError("Search supports FixedPred or Strict")

    local_loaders = build_dataloaders(
        dataset_name,
        split_seed=CFG.split_seed,
        loader_seed=CFG.loader_seed,
        train_total_subset=train_total_subset,
        test_subset=None,
        include_test=False,
    )
    factory = MODEL_FACTORIES[model_name]
    rows = []

    for cfg in grid:
        eta, n = float(cfg["eta"]), int(cfg["n"])
        run_id = f"pilot_{dataset_name}_{model_name}_{method}_seed{seed}_eta{safe_name(eta)}_n{n}"
        checkpoint = CHECKPOINT_DIR / f"{run_id}.pt"
        try:
            model, history, metadata = fit_with_validation(
                factory,
                local_loaders["train"],
                local_loaders["validation"],
                method=method,
                seed=seed,
                epochs=epochs,
                lr=CFG.learning_rate,
                eta=eta,
                n=n,
                checkpoint_path=checkpoint,
                max_batches=max_batches,
            )
            best = history.loc[history["val_macro_f1"].idxmax()].to_dict()
            rows.append({
                "run_id": run_id,
                "dataset": dataset_name,
                "model": model_name,
                "method": method,
                "seed": seed,
                "eta": eta,
                "n": n,
                "status": "ok",
                "checkpoint": str(checkpoint),
                **best,
            })
            history.to_csv(METRIC_DIR / f"{run_id}_history.csv", index=False)
        except Exception as exc:
            rows.append({
                "run_id": run_id,
                "dataset": dataset_name,
                "model": model_name,
                "method": method,
                "seed": seed,
                "eta": eta,
                "n": n,
                "status": "failed",
                "error": repr(exc),
            })

    return pd.DataFrame(rows)


def aggregate_pilot_results(search_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["dataset", "model", "method", "eta", "n"]
    for keys, group in search_df.groupby(group_cols, dropna=False):
        ok = group[group["status"] == "ok"].copy()
        attempts = int(group["seed"].nunique())
        successes = int(ok["seed"].nunique())
        row = dict(zip(group_cols, keys))
        row.update({
            "attempts": attempts,
            "successes": successes,
            "success_rate": successes / max(attempts, 1),
            "mean_val_macro_f1": float(ok["val_macro_f1"].mean()) if not ok.empty else float("nan"),
            "std_val_macro_f1": float(ok["val_macro_f1"].std(ddof=1)) if successes > 1 else 0.0,
            "mean_epoch_time_sec": float(ok["epoch_time_sec"].mean()) if not ok.empty else float("inf"),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def select_config(search_df: pd.DataFrame) -> dict[str, Any]:
    summary = aggregate_pilot_results(search_df)
    eligible = summary[
        (summary["success_rate"] >= CFG.pilot_min_success_rate)
        & np.isfinite(summary["mean_val_macro_f1"])
    ].copy()
    if eligible.empty:
        raise RuntimeError("No configuration reached the required pilot success rate")
    eligible = eligible.sort_values(
        ["mean_val_macro_f1", "success_rate", "std_val_macro_f1", "mean_epoch_time_sec", "n"],
        ascending=[False, False, True, True, True],
    )
    row = eligible.iloc[0]
    return {
        "method": row["method"],
        "eta": float(row["eta"]),
        "n": int(row["n"]),
        "pilot_seed_count": int(row["attempts"]),
        "pilot_success_rate": float(row["success_rate"]),
        "pilot_mean_val_macro_f1": float(row["mean_val_macro_f1"]),
        "pilot_std_val_macro_f1": float(row["std_val_macro_f1"]),
        "selected_by": "mean_validation_macro_f1_then_success_rate_stability_time",
    }

pilot_frames = []
if RUN_PILOT_SEARCH:
    pilot_subset = 12000 if CFG.run_mode != "smoke" else 1200
    for dataset_name in active_dataset_names():
        for pilot_seed in CFG.pilot_seeds:
            pilot_frames.append(run_pc_search(
                dataset_name, "lenet_classic", "FixedPred", CFG.fixedpred_grid,
                seed=pilot_seed, epochs=CFG.pilot_epochs,
                train_total_subset=pilot_subset,
                max_batches=10 if CFG.run_mode == "smoke" else None,
            ))
            pilot_frames.append(run_pc_search(
                dataset_name, "lenet_classic", "Strict", CFG.strict_grid,
                seed=pilot_seed, epochs=CFG.pilot_epochs,
                train_total_subset=pilot_subset,
                max_batches=10 if CFG.run_mode == "smoke" else None,
            ))

pilot_results = pd.concat(pilot_frames, ignore_index=True) if pilot_frames else pd.DataFrame()
if not pilot_results.empty:
    pilot_results.to_csv(METRIC_DIR / "pilot_search_results.csv", index=False)
    display(pilot_results)
    display(aggregate_pilot_results(pilot_results))

In [ ]:
# 12. Фиксация выбранных конфигураций

SELECTED_CONFIG_PATH = CONFIG_DIR / "selected_configs.json"

if RUN_PILOT_SEARCH and not pilot_results.empty:
    selections = {}
    for dataset_name in active_dataset_names():
        selections[dataset_name] = {}
        for method in ["FixedPred", "Strict"]:
            subset = pilot_results[
                (pilot_results["dataset"] == dataset_name)
                & (pilot_results["method"] == method)
            ]
            selections[dataset_name][method] = select_config(subset)
    SELECTED_CONFIG_PATH.write_text(
        json.dumps(selections, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Saved:", SELECTED_CONFIG_PATH)
elif SELECTED_CONFIG_PATH.exists():
    selections = json.loads(SELECTED_CONFIG_PATH.read_text(encoding="utf-8"))
    print("Loaded selections:", selections)
else:
    selections = {
        dataset_name: {
            "FixedPred": {"method": "FixedPred", "eta": 0.3, "n": 10, "selected_by": "placeholder"},
            "Strict": {"method": "Strict", "eta": 0.05, "n": 20, "selected_by": "placeholder"},
        }
        for dataset_name in active_dataset_names()
    }
    print("Placeholder selections are active. Run pilot before final experiments.")

## 7. Финальное многостартовое сравнение

Финальная функция обучает по train, выбирает checkpoint по validation и только затем один раз оценивает test.

In [ ]:
# 13. Финальные запуски

def final_run(
    dataset_name: str,
    model_name: str,
    method: str,
    seed: int,
    eta: Optional[float],
    n: Optional[int],
    epochs: int,
    selection_origin: Optional[str] = None,
    optimizer_name: Optional[str] = None,
    learning_rate: Optional[float] = None,
    split_seed_value: Optional[int] = None,
    loader_seed_value: Optional[int] = None,
    run_prefix: str = "final",
) -> tuple[dict[str, Any], Path]:
    split_seed_value = CFG.split_seed if split_seed_value is None else int(split_seed_value)
    loader_seed_value = CFG.loader_seed if loader_seed_value is None else int(loader_seed_value)
    optimizer_name = CFG.optimizer_name if optimizer_name is None else optimizer_name
    learning_rate = (
        CFG.sgd_learning_rate
        if learning_rate is None and optimizer_name.lower() == "sgd"
        else CFG.learning_rate if learning_rate is None else float(learning_rate)
    )
    local_loaders = build_dataloaders(
        dataset_name,
        split_seed=split_seed_value,
        loader_seed=loader_seed_value,
    )
    factory = MODEL_FACTORIES[model_name]
    run_id = f"{run_prefix}_{dataset_name}_{model_name}_{method}_seed{seed}_opt{optimizer_name}"
    if selection_origin:
        run_id += f"_selected_on_{selection_origin}"
    if eta is not None:
        run_id += f"_eta{safe_name(eta)}_n{n}"
    checkpoint = CHECKPOINT_DIR / f"{run_id}.pt"

    model, history, metadata = fit_with_validation(
        factory,
        local_loaders["train"],
        local_loaders["validation"],
        method=method,
        seed=seed,
        epochs=epochs,
        lr=learning_rate,
        optimizer_name=optimizer_name,
        split_seed_value=split_seed_value,
        loader_seed_value=loader_seed_value,
        eta=eta,
        n=n,
        checkpoint_path=checkpoint,
    )
    test_metrics = evaluate_classifier(model, local_loaders["test"])
    history_path = METRIC_DIR / f"{run_id}_history.csv"
    history.to_csv(history_path, index=False)
    row = {
        "run_id": run_id,
        "dataset": dataset_name,
        "model": model_name,
        "method": method,
        "seed": seed,
        "model_seed": seed,
        "split_seed": split_seed_value,
        "loader_seed": loader_seed_value,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "selection_origin": selection_origin or dataset_name,
        "eta": eta,
        "n": n,
        "best_epoch": metadata["best_epoch"],
        "best_validation_metric": metadata["best_validation_metric"],
        "test_loss": test_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "checkpoint": str(checkpoint),
        "history_path": str(history_path),
        "trained_epochs": int(len(history)),
        "total_train_time_sec": float(history["epoch_time_sec"].sum()),
        "torch2pc_commit": TORCH2PC_HEAD,
        "status": "ok",
    }
    return row, checkpoint


def run_final_suite() -> pd.DataFrame:
    if not SELECTED_CONFIG_PATH.exists():
        raise FileNotFoundError(
            "Run pilot and save selected_configs.json before final suite"
        )
    selected = json.loads(SELECTED_CONFIG_PATH.read_text(encoding="utf-8"))
    rows = []
    for dataset_name in active_dataset_names():
        methods = [
            ("BP", None, None),
            (
                "FixedPred",
                selected[dataset_name]["FixedPred"]["eta"],
                selected[dataset_name]["FixedPred"]["n"],
            ),
            (
                "Strict",
                selected[dataset_name]["Strict"]["eta"],
                selected[dataset_name]["Strict"]["n"],
            ),
        ]
        for seed in CFG.final_seeds:
            for method, eta, n in methods:
                try:
                    row, _ = final_run(
                        dataset_name,
                        "lenet_classic",
                        method,
                        seed,
                        eta,
                        n,
                        CFG.final_epochs,
                        selection_origin=dataset_name,
                    )
                except Exception as exc:
                    row = {
                        "dataset": dataset_name,
                        "model": "lenet_classic",
                        "method": method,
                        "seed": seed,
                        "model_seed": seed,
                        "split_seed": CFG.split_seed,
                        "loader_seed": CFG.loader_seed,
                        "selection_origin": dataset_name,
                        "eta": eta,
                        "n": n,
                        "status": "failed",
                        "error": repr(exc),
                    }
                rows.append(row)
    return pd.DataFrame(rows)

final_results = run_final_suite() if RUN_FINAL_EXPERIMENTS else pd.DataFrame()
if not final_results.empty:
    final_results.to_csv(METRIC_DIR / "final_results_by_seed.csv", index=False)
    display(final_results)

## 7A. H4: сложность данных и перенос конфигурации

H4 проверяется двумя взаимодополняющими способами.

1. **Настроенное сравнение:** для каждого набора данных используется конфигурация, выбранная только по его валидационной выборке.
2. **Перенос конфигурации:** значения `eta` и `n`, выбранные на MNIST, без повторного поиска применяются при обучении на Fashion-MNIST.

Первый анализ показывает качество лучшего найденного режима в рамках одинакового бюджета поиска. Второй отделяет влияние сложности данных от повторной настройки гиперпараметров. Тестовая выборка не участвует в выборе конфигурации.

In [ ]:
# 13A. H4: перенос фиксированной конфигурации между наборами данных

def run_dataset_transfer_suite(
    selected: dict[str, Any],
    source_dataset: str,
    target_dataset: str,
    seeds: list[int],
) -> pd.DataFrame:
    if source_dataset not in selected:
        raise KeyError(f"No selected configuration for {source_dataset}")

    rows = []
    for method in ["FixedPred", "Strict"]:
        source_cfg = selected[source_dataset][method]
        for seed in seeds:
            try:
                row, _ = final_run(
                    target_dataset,
                    "lenet_classic",
                    method,
                    seed,
                    float(source_cfg["eta"]),
                    int(source_cfg["n"]),
                    CFG.final_epochs,
                    selection_origin=source_dataset,
                )
                row["analysis"] = "H4_fixed_configuration_transfer"
                row["source_dataset"] = source_dataset
                row["target_dataset"] = target_dataset
            except Exception as exc:
                row = {
                    "analysis": "H4_fixed_configuration_transfer",
                    "source_dataset": source_dataset,
                    "target_dataset": target_dataset,
                    "dataset": target_dataset,
                    "model": "lenet_classic",
                    "method": method,
                    "seed": seed,
                    "model_seed": seed,
                    "split_seed": CFG.split_seed,
                    "loader_seed": CFG.loader_seed,
                    "selection_origin": source_dataset,
                    "eta": source_cfg["eta"],
                    "n": source_cfg["n"],
                    "status": "failed",
                    "error": repr(exc),
                }
            rows.append(row)
    return pd.DataFrame(rows)


dataset_transfer_results = pd.DataFrame()
if RUN_DATASET_TRANSFER_ANALYSIS:
    if not SELECTED_CONFIG_PATH.exists():
        raise FileNotFoundError("selected_configs.json is required")
    selected_all = json.loads(
        SELECTED_CONFIG_PATH.read_text(encoding="utf-8")
    )
    dataset_transfer_results = run_dataset_transfer_suite(
        selected_all,
        CFG.h4_source_dataset,
        CFG.h4_target_dataset,
        CFG.final_seeds,
    )
    dataset_transfer_results.to_csv(
        METRIC_DIR / "h4_dataset_transfer_results.csv",
        index=False,
    )
    display(dataset_transfer_results)

## 8. Послойное сравнение обученных checkpoints

In [ ]:
# 14. Активации, CKA и RSA Spearman

def build_stratified_analysis_loader(
    dataset_name: str,
    max_samples: Optional[int] = None,
    sample_seed: Optional[int] = None,
) -> DataLoader:
    max_samples = CFG.representation_max_samples if max_samples is None else int(max_samples)
    sample_seed = CFG.representation_sample_seed if sample_seed is None else int(sample_seed)
    cls = dataset_class(dataset_name)
    kwargs = {"split": "digits"} if dataset_name == "EMNIST" else {}
    dataset = cls(
        root="data", train=False, download=True,
        transform=image_transform(), **kwargs
    )
    targets = dataset_targets(dataset)
    indices = stratified_subset_indices(targets, max_samples, sample_seed)
    index_path = SPLIT_DIR / f"{dataset_name}_representation_seed{sample_seed}_n{len(indices)}.npy"
    np.save(index_path, indices)
    return DataLoader(
        Subset(dataset, indices.tolist()),
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=DEVICE.type == "cuda",
        worker_init_fn=seed_worker,
        persistent_workers=CFG.num_workers > 0,
    )


def collect_layer_activations(
    model: nn.Sequential,
    loader: DataLoader,
    max_samples: int = 512,
) -> dict[str, torch.Tensor]:
    model.eval()
    collected: dict[str, list[torch.Tensor]] = {str(i): [] for i in range(len(model))}
    hooks = []

    for i, layer in enumerate(model):
        def hook_factory(name: str):
            def hook(_module, _inputs, output):
                tensor = output[0] if isinstance(output, tuple) else output
                collected[name].append(tensor.detach().flatten(1).cpu())
            return hook
        hooks.append(layer.register_forward_hook(hook_factory(str(i))))

    seen = 0
    with torch.no_grad():
        for xb, _ in loader:
            if seen >= max_samples:
                break
            take = min(xb.shape[0], max_samples - seen)
            model(xb[:take].to(DEVICE))
            seen += take

    for hook in hooks:
        hook.remove()
    return {name: torch.cat(parts, dim=0) for name, parts in collected.items() if parts}


def center_features(x: torch.Tensor) -> torch.Tensor:
    x = x.float()
    return x - x.mean(dim=0, keepdim=True)


def linear_cka(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-12) -> float:
    x, y = center_features(x), center_features(y)
    hsic = torch.norm(x.T @ y, p="fro") ** 2
    norm_x = torch.norm(x.T @ x, p="fro")
    norm_y = torch.norm(y.T @ y, p="fro")
    return float(hsic / (norm_x * norm_y + eps))


def upper_triangle_vector(matrix: torch.Tensor) -> np.ndarray:
    idx = torch.triu_indices(matrix.shape[0], matrix.shape[1], offset=1)
    return matrix[idx[0], idx[1]].cpu().numpy()


def rsa_spearman(x: torch.Tensor, y: torch.Tensor) -> float:
    dx = torch.cdist(x.float(), x.float(), p=2)
    dy = torch.cdist(y.float(), y.float(), p=2)
    result = stats.spearmanr(upper_triangle_vector(dx), upper_triangle_vector(dy))
    return float(result.statistic)


def representation_matrix(
    activations_a: dict[str, torch.Tensor],
    activations_b: dict[str, torch.Tensor],
    metric: str,
) -> pd.DataFrame:
    rows = []
    for layer_a, xa in activations_a.items():
        for layer_b, yb in activations_b.items():
            n_samples = min(xa.shape[0], yb.shape[0])
            if metric == "cka":
                value = linear_cka(xa[:n_samples], yb[:n_samples])
            elif metric == "rsa":
                value = rsa_spearman(xa[:n_samples], yb[:n_samples])
            else:
                raise ValueError(metric)
            rows.append({"layer_a": layer_a, "layer_b": layer_b, metric: value})
    return pd.DataFrame(rows)

def bootstrap_representation_similarity(
    x: torch.Tensor,
    y: torch.Tensor,
    metric: str,
    repeats: Optional[int] = None,
    seed: Optional[int] = None,
) -> dict[str, float]:
    repeats = CFG.representation_bootstrap_repeats if repeats is None else int(repeats)
    seed = CFG.representation_sample_seed if seed is None else int(seed)
    n = min(int(x.shape[0]), int(y.shape[0]))
    x, y = x[:n], y[:n]
    bootstrap_size = min(n, int(CFG.representation_bootstrap_sample_size))
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(repeats):
        idx = torch.from_numpy(rng.integers(0, n, size=bootstrap_size)).long()
        if metric == "cka":
            value = linear_cka(x[idx], y[idx])
        elif metric == "rsa":
            value = rsa_spearman(x[idx], y[idx])
        else:
            raise ValueError(metric)
        if np.isfinite(value):
            values.append(float(value))
    if not values:
        return {"estimate": float("nan"), "ci95_low": float("nan"), "ci95_high": float("nan"), "bootstrap_repeats": 0}
    return {
        "estimate": float(np.mean(values)),
        "ci95_low": float(np.quantile(values, 0.025)),
        "ci95_high": float(np.quantile(values, 0.975)),
        "bootstrap_repeats": len(values),
        "bootstrap_sample_size": bootstrap_size,
    }


def representation_diagonal_bootstrap(
    activations_a: dict[str, torch.Tensor],
    activations_b: dict[str, torch.Tensor],
) -> pd.DataFrame:
    rows = []
    for layer in sorted(activations_a.keys() & activations_b.keys(), key=int):
        for metric in ["cka", "rsa"]:
            result = bootstrap_representation_similarity(
                activations_a[layer], activations_b[layer], metric,
                seed=stable_int_seed(CFG.representation_sample_seed, layer, metric),
            )
            rows.append({"layer": layer, "metric": metric, **result})
    return pd.DataFrame(rows)


In [ ]:
# 15. Выходное поведение

def collect_probabilities(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    metrics = evaluate_classifier(model, loader)
    return metrics["probabilities"], metrics["y_true"]


def symmetric_kl(p: np.ndarray, q: np.ndarray, eps: float = 1e-8) -> float:
    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)
    kl_pq = np.sum(p * np.log(p / q), axis=1)
    kl_qp = np.sum(q * np.log(q / p), axis=1)
    return float(np.mean(0.5 * (kl_pq + kl_qp)))


def output_comparison(model_a: nn.Module, model_b: nn.Module, loader: DataLoader) -> dict[str, float]:
    p, y = collect_probabilities(model_a, loader)
    q, _ = collect_probabilities(model_b, loader)
    pred_a, pred_b = p.argmax(axis=1), q.argmax(axis=1)
    confidence_a, confidence_b = p.max(axis=1), q.max(axis=1)
    return {
        "agreement": float(np.mean(pred_a == pred_b)),
        "symmetric_kl": symmetric_kl(p, q),
        "mean_confidence_gap": float(np.mean(np.abs(confidence_a - confidence_b))),
        "accuracy_a": float(np.mean(pred_a == y)),
        "accuracy_b": float(np.mean(pred_b == y)),
    }


def compare_trained_checkpoints(
    checkpoint_a: Path,
    checkpoint_b: Path,
    model_factory: Callable[[], nn.Sequential],
    loader: DataLoader,
    label_a: str,
    label_b: str,
) -> dict[str, pd.DataFrame | dict[str, float]]:
    model_a, _ = load_checkpoint(model_factory, checkpoint_a)
    model_b, _ = load_checkpoint(model_factory, checkpoint_b)
    act_a = collect_layer_activations(model_a, loader)
    act_b = collect_layer_activations(model_b, loader)
    return {
        "cka": representation_matrix(act_a, act_b, "cka"),
        "rsa": representation_matrix(act_a, act_b, "rsa"),
        "outputs": output_comparison(model_a, model_b, loader),
        "labels": {"a": label_a, "b": label_b},
    }

## 9. Устойчивость финальных моделей

In [ ]:
# 16. Детерминированные искажения

def corruption_strength(severity: int) -> float:
    if severity not in {1, 2, 3, 4, 5}:
        raise ValueError("severity must be 1..5")
    return severity / 5.0


def corruption_batch_seed(
    dataset_name: str,
    corruption: str,
    severity: int,
    batch_index: int,
) -> int:
    return stable_int_seed(
        CFG.corruption_seed,
        dataset_name,
        corruption,
        severity,
        batch_index,
    )


def tensor_generator(x: torch.Tensor, seed: int) -> torch.Generator:
    generator = torch.Generator(device=x.device)
    generator.manual_seed(int(seed))
    return generator


def corrupt_batch(
    x: torch.Tensor,
    corruption: str,
    severity: int,
    seed: Optional[int] = None,
) -> torch.Tensor:
    strength = corruption_strength(severity)
    x = x.clone()
    generator = tensor_generator(
        x,
        CFG.corruption_seed if seed is None else seed,
    )

    if corruption == "clean":
        return x
    if corruption == "gaussian_noise":
        noise = torch.randn(
            x.shape,
            generator=generator,
            device=x.device,
            dtype=x.dtype,
        )
        return torch.clamp(
            x + noise * (0.1 + 0.4 * strength),
            0,
            1,
        )
    if corruption == "salt_pepper":
        probability = 0.02 + 0.18 * strength
        mask = torch.rand(
            x.shape,
            generator=generator,
            device=x.device,
            dtype=x.dtype,
        )
        x[mask < probability / 2] = 0
        x[(mask >= probability / 2) & (mask < probability)] = 1
        return x
    if corruption == "pixel_dropout":
        probability = 0.05 + 0.45 * strength
        mask = torch.rand(
            x.shape,
            generator=generator,
            device=x.device,
            dtype=x.dtype,
        )
        return x * (mask > probability)
    if corruption == "low_contrast":
        factor = 1.0 - 0.8 * strength
        mean = x.mean(dim=(-2, -1), keepdim=True)
        return torch.clamp(mean + factor * (x - mean), 0, 1)
    if corruption == "occlusion":
        side = int(round(4 + 12 * strength))
        h, w = x.shape[-2:]
        top = max(0, (h - side) // 2)
        left = max(0, (w - side) // 2)
        x[..., top:top + side, left:left + side] = 0
        return x
    if corruption == "rotation":
        angle = 6 + 24 * strength
        return torch.stack(
            [transforms.functional.rotate(sample, angle) for sample in x]
        )
    if corruption == "gaussian_blur":
        kernel = 3 if severity <= 2 else 5 if severity <= 4 else 7
        sigma = 0.3 + 1.7 * strength
        return transforms.functional.gaussian_blur(
            x,
            kernel_size=[kernel, kernel],
            sigma=[sigma, sigma],
        )
    raise ValueError(f"Unknown corruption: {corruption}")


def active_corruptions() -> list[str]:
    selected = (
        CFG.core_corruptions
        if CFG.analysis_tier == "core"
        else CFG.extended_corruptions
    )
    return ["clean", *selected]


def evaluate_corrupted(
    model: nn.Module,
    loader: DataLoader,
    dataset_name: str,
    corruption: str,
    severity: int,
) -> dict[str, float]:
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    y_true, y_pred = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch_index, (xb, yb) in enumerate(loader):
            seed = corruption_batch_seed(
                dataset_name, corruption, severity, batch_index
            )
            xb = corrupt_batch(
                xb, corruption, severity, seed=seed
            ).to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            ensure_finite_tensor("corrupted_logits", logits)
            batch_loss = loss_fn(logits, yb)
            ensure_finite_tensor("corrupted_loss", batch_loss)
            total_loss += float(batch_loss.item())
            y_true.extend(yb.cpu().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())
    if not y_true:
        raise RuntimeError("Corrupted evaluation produced no samples")
    return {
        "loss": total_loss / len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "corruption_seed_base": CFG.corruption_seed,
    }


corruption_protocol = {
    "base_seed": CFG.corruption_seed,
    "seed_formula": (
        "sha256(base_seed|dataset|corruption|severity|batch_index)"
    ),
    "paired_across_methods": True,
    "core_corruptions": CFG.core_corruptions,
    "extended_corruptions": CFG.extended_corruptions,
}
(CONFIG_DIR / "corruption_protocol.json").write_text(
    json.dumps(corruption_protocol, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

## 10. Ошибки предсказания по объектам и слоям

Функция использует обученную модель. Потеря классификации рассчитывается отдельно для каждого объекта.

In [ ]:
# 17. Ошибки предсказания по объектам и слоям

def epsilon_per_sample(
    epsilon: Iterable[Optional[torch.Tensor]],
    batch_size: int,
) -> pd.DataFrame:
    rows = []
    for layer_idx, eps in enumerate(epsilon):
        if eps is None:
            continue
        tensor = eps.detach()
        ensure_finite_tensor(f"epsilon_layer_{layer_idx}", tensor)
        if tensor.ndim == 0 or tensor.shape[0] != batch_size:
            value = float(torch.sqrt(torch.mean(tensor.float() ** 2)))
            for sample_idx in range(batch_size):
                rows.append({
                    "sample_in_batch": sample_idx,
                    "layer": layer_idx,
                    "epsilon_rms": value,
                })
            continue
        flat = tensor.float().reshape(batch_size, -1)
        rms = torch.sqrt(torch.mean(flat ** 2, dim=1)).cpu().numpy()
        for sample_idx, value in enumerate(rms):
            rows.append({
                "sample_in_batch": sample_idx,
                "layer": layer_idx,
                "epsilon_rms": float(value),
            })
    return pd.DataFrame(rows)


def prediction_error_diagnostics(
    model: nn.Module,
    loader: DataLoader,
    dataset_name: str,
    method: str,
    eta: float,
    n: int,
    corruption: str = "clean",
    severity: int = 1,
    max_batches: int = 5,
) -> pd.DataFrame:
    model.eval()
    loss_none = nn.CrossEntropyLoss(reduction="none")
    loss_mean = nn.CrossEntropyLoss()
    rows = []
    global_index = 0

    for batch_index, (xb, yb) in enumerate(loader):
        if batch_index >= max_batches:
            break
        batch_corruption_seed = corruption_batch_seed(
            dataset_name,
            corruption,
            severity,
            batch_index,
        )
        xb = corrupt_batch(
            xb,
            corruption,
            severity,
            seed=batch_corruption_seed,
        ).to(DEVICE)
        yb = yb.to(DEVICE)
        batch_size = int(yb.shape[0])

        with torch.no_grad():
            logits = model(xb)
            ensure_finite_tensor("diagnostic_logits", logits)
            probability = torch.softmax(logits, dim=1)
            prediction = probability.argmax(dim=1)
            confidence = probability.max(dim=1).values
            sample_loss = loss_none(logits, yb)
            ensure_finite_tensor("per_sample_loss", sample_loss)

        model.zero_grad(set_to_none=True)
        _, pc_loss, _, _, epsilon = PCInfer(
            model,
            loss_mean,
            xb,
            yb,
            method,
            eta=float(eta),
            n=int(n),
        )
        ensure_finite_tensor("diagnostic_pc_loss", pc_loss)
        eps_df = epsilon_per_sample(epsilon, batch_size)

        metadata = pd.DataFrame({
            "sample_in_batch": np.arange(batch_size),
            "sample_id": np.arange(
                global_index,
                global_index + batch_size,
            ),
            "target": yb.detach().cpu().numpy(),
            "prediction": prediction.detach().cpu().numpy(),
            "correct": (
                prediction == yb
            ).detach().cpu().numpy().astype(int),
            "confidence": confidence.detach().cpu().numpy(),
            "classification_loss": sample_loss.detach().cpu().numpy(),
            "corruption": corruption,
            "severity": severity,
            "corruption_batch_seed": batch_corruption_seed,
            "method": method,
            "eta": eta,
            "n": n,
        })
        rows.append(
            eps_df.merge(
                metadata,
                on="sample_in_batch",
                how="left",
            )
        )
        global_index += batch_size

    return (
        pd.concat(rows, ignore_index=True)
        if rows
        else pd.DataFrame()
    )

## 11. Измерение времени

Обычное прогнозирование, BP step, PC step и диагностический PCInfer измеряются отдельно.

In [ ]:
# 18. Временные измерения

def benchmark_callable(fn: Callable[[], Any], warmup: int = 3, repeats: int = 10) -> dict[str, float]:
    for _ in range(warmup):
        fn()
    sync_device()
    times = []
    peak_memory = []
    for _ in range(repeats):
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        fn()
        sync_device()
        times.append(time.perf_counter() - start)
        if DEVICE.type == "cuda":
            peak_memory.append(float(torch.cuda.max_memory_allocated()))
    return {
        "mean_sec": float(np.mean(times)),
        "std_sec": float(np.std(times, ddof=1)) if len(times) > 1 else 0.0,
        "median_sec": float(np.median(times)),
        "iqr_sec": float(np.quantile(times, 0.75) - np.quantile(times, 0.25)),
        "peak_memory_bytes": float(max(peak_memory)) if peak_memory else float("nan"),
    }


def timing_suite(model: nn.Module, xb: torch.Tensor, yb: torch.Tensor, method: str, eta: Optional[float], n: Optional[int]) -> pd.DataFrame:
    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()

    forward_model = copy.deepcopy(model).to(DEVICE).eval()
    bp_model = copy.deepcopy(model).to(DEVICE).train()
    bp_optimizer = make_optimizer(bp_model, CFG.optimizer_name, CFG.learning_rate)

    def forward_only():
        with torch.no_grad():
            forward_model(xb)

    def bp_step():
        bp_optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(bp_model(xb), yb)
        loss.backward()
        bp_optimizer.step()

    rows = []
    for label, fn in [("forward_prediction", forward_only), ("bp_training_step", bp_step)]:
        result = benchmark_callable(fn)
        rows.append({"operation": label, "method": "BP", "batch_size": len(yb), **result})

    if method in {"FixedPred", "Strict"}:
        pc_model = copy.deepcopy(model).to(DEVICE).train()
        pc_optimizer = make_optimizer(pc_model, CFG.optimizer_name, CFG.learning_rate)

        def pc_step():
            pc_optimizer.zero_grad(set_to_none=True)
            PCInfer(pc_model, loss_fn, xb, yb, method, eta=float(eta), n=int(n))
            pc_optimizer.step()

        result = benchmark_callable(pc_step)
        rows.append({
            "operation": "pc_training_step",
            "method": method,
            "eta": eta,
            "n": n,
            "batch_size": len(yb),
            **result,
        })

    for row in rows:
        row["mean_sec_per_sample"] = row["mean_sec"] / row["batch_size"]
    return pd.DataFrame(rows)

## 11A. Оркестрация диагностических анализов

Следующие функции загружают только сохраненные best-validation checkpoints. Они не создают новые случайно инициализированные модели для научной интерпретации.

In [ ]:
# 18A. Загрузка таблиц и сравнение параметров/обновлений

def load_results_table(current: pd.DataFrame, path: Path) -> pd.DataFrame:
    if current is not None and not current.empty:
        return current.copy()
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def named_parameter_vectors(model: nn.Module) -> dict[str, torch.Tensor]:
    return {
        name: parameter.detach().flatten().cpu()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }


def compare_trained_updates(
    model_a: nn.Module,
    model_b: nn.Module,
    loader: DataLoader,
    method_a: str,
    method_b: str,
    eta_a: Optional[float] = None,
    n_a: Optional[int] = None,
    eta_b: Optional[float] = None,
    n_b: Optional[int] = None,
    max_batches: Optional[int] = None,
) -> pd.DataFrame:
    max_batches = CFG.gradient_batches if max_batches is None else int(max_batches)
    frames = []
    for batch_index, (xb, yb) in enumerate(loader):
        if batch_index >= max_batches:
            break
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        grad_a, _ = gradient_for_method(copy.deepcopy(model_a), xb, yb, method_a, eta_a, n_a)
        grad_b, _ = gradient_for_method(copy.deepcopy(model_b), xb, yb, method_b, eta_b, n_b)
        table = compare_gradient_maps(grad_a, grad_b)
        table.insert(0, "batch_index", batch_index)
        frames.append(table)
    if not frames:
        raise RuntimeError("No batches were available for gradient comparison")
    return pd.concat(frames, ignore_index=True)


def aligned_activation_metrics(
    activations_a: dict[str, torch.Tensor],
    activations_b: dict[str, torch.Tensor],
) -> pd.DataFrame:
    rows = []
    for layer in sorted(activations_a.keys() & activations_b.keys(), key=int):
        xa, xb = activations_a[layer].float(), activations_b[layer].float()
        n_samples = min(xa.shape[0], xb.shape[0])
        xa, xb = xa[:n_samples], xb[:n_samples]
        if xa.shape[1] != xb.shape[1]:
            continue
        sample_cosine = torch.nn.functional.cosine_similarity(xa, xb, dim=1)
        rows.append({
            "layer": layer,
            "activation_mse": float(torch.mean((xa - xb) ** 2)),
            "activation_cosine_mean": float(sample_cosine.mean()),
            "activation_cosine_std": float(sample_cosine.std(unbiased=True)) if n_samples > 1 else 0.0,
        })
    return pd.DataFrame(rows)

def run_layerwise_suite(results_df: pd.DataFrame, max_samples: int = CFG.representation_max_samples) -> dict[str, pd.DataFrame]:
    cka_rows, rsa_rows, bootstrap_rows, activation_rows, output_rows, update_rows, parameter_rows = [], [], [], [], [], [], []
    ok = results_df[results_df["status"] == "ok"].copy()

    for (dataset_name, model_name, seed), group in ok.groupby(["dataset", "model", "seed"]):
        bp_rows = group[group["method"] == "BP"]
        if bp_rows.empty:
            continue
        bp_row = bp_rows.iloc[0]
        bp_model, _ = load_checkpoint(MODEL_FACTORIES[model_name], Path(bp_row["checkpoint"]))
        local_loaders = build_dataloaders(
            dataset_name,
            split_seed=CFG.split_seed,
            loader_seed=CFG.loader_seed,
        )

        analysis_loader = build_stratified_analysis_loader(
            dataset_name,
            max_samples=max_samples,
            sample_seed=CFG.representation_sample_seed,
        )

        for method in ["FixedPred", "Strict"]:
            candidate_rows = group[group["method"] == method]
            if candidate_rows.empty:
                continue
            candidate_row = candidate_rows.iloc[0]
            candidate_model, _ = load_checkpoint(
                MODEL_FACTORIES[model_name], Path(candidate_row["checkpoint"])
            )

            act_bp = collect_layer_activations(bp_model, analysis_loader, max_samples=max_samples)
            act_pc = collect_layer_activations(candidate_model, analysis_loader, max_samples=max_samples)
            cka = representation_matrix(act_bp, act_pc, "cka")
            rsa = representation_matrix(act_bp, act_pc, "rsa")
            activations = aligned_activation_metrics(act_bp, act_pc)
            bootstrap = representation_diagonal_bootstrap(act_bp, act_pc) if RUN_REPRESENTATION_BOOTSTRAP else pd.DataFrame()
            outputs = output_comparison(bp_model, candidate_model, local_loaders["test"])
            updates_by_state = []
            for state_label, state_model in [("bp_checkpoint", bp_model), ("candidate_checkpoint", candidate_model)]:
                state_updates = compare_trained_updates(
                    state_model,
                    state_model,
                    local_loaders["validation"],
                    method_a="BP",
                    method_b=method,
                    eta_b=float(candidate_row["eta"]),
                    n_b=int(candidate_row["n"]),
                    max_batches=CFG.gradient_batches,
                )
                state_updates.insert(0, "shared_parameter_state", state_label)
                updates_by_state.append(state_updates)
            updates = pd.concat(updates_by_state, ignore_index=True)
            parameters = compare_gradient_maps(
                named_parameter_vectors(bp_model), named_parameter_vectors(candidate_model)
            )

            common = {
                "dataset": dataset_name,
                "model": model_name,
                "seed": int(seed),
                "reference": "BP",
                "candidate": method,
                "eta": candidate_row.get("eta"),
                "n": candidate_row.get("n"),
            }
            for table, sink in [(cka, cka_rows), (rsa, rsa_rows)]:
                for row in table.to_dict("records"):
                    sink.append({**common, **row})
            for row in activations.to_dict("records"):
                activation_rows.append({**common, **row})
            for row in bootstrap.to_dict("records"):
                bootstrap_rows.append({**common, **row})
            output_rows.append({**common, **outputs})
            for row in updates.to_dict("records"):
                update_rows.append({**common, **row})
            for row in parameters.to_dict("records"):
                parameter_rows.append({**common, **row})

    return {
        "cka": pd.DataFrame(cka_rows),
        "rsa": pd.DataFrame(rsa_rows),
        "representation_bootstrap": pd.DataFrame(bootstrap_rows),
        "activations": pd.DataFrame(activation_rows),
        "outputs": pd.DataFrame(output_rows),
        "updates": pd.DataFrame(update_rows),
        "parameters": pd.DataFrame(parameter_rows),
    }

resolved_final_results = load_results_table(
    final_results, METRIC_DIR / "final_results_by_seed.csv"
)
layerwise_tables = {}
if RUN_LAYERWISE_ANALYSIS:
    if resolved_final_results.empty:
        raise FileNotFoundError("Final results are required for layer-wise analysis")
    layerwise_tables = run_layerwise_suite(resolved_final_results)
    for name, table in layerwise_tables.items():
        table.to_csv(METRIC_DIR / f"layerwise_{name}.csv", index=False)
        print(name, table.shape)

In [ ]:
# 18B. Устойчивость, ошибки предсказания и время на финальных checkpoints

def run_robustness_suite(results_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    ok = results_df[results_df["status"] == "ok"].copy()
    for _, row in ok.iterrows():
        model, _ = load_checkpoint(
            MODEL_FACTORIES[row["model"]],
            Path(row["checkpoint"]),
        )
        test_loader = build_dataloaders(
            row["dataset"],
            split_seed=CFG.split_seed,
            loader_seed=CFG.loader_seed,
        )["test"]
        for corruption in active_corruptions():
            severities = [1] if corruption == "clean" else [1, 2, 3, 4, 5]
            for severity in severities:
                metrics = evaluate_corrupted(
                    model,
                    test_loader,
                    row["dataset"],
                    corruption,
                    severity,
                )
                rows.append({
                    "dataset": row["dataset"],
                    "model": row["model"],
                    "method": row["method"],
                    "seed": int(row["seed"]),
                    "model_seed": int(row["seed"]),
                    "split_seed": CFG.split_seed,
                    "eta": row.get("eta"),
                    "n": row.get("n"),
                    "corruption": corruption,
                    "severity": severity,
                    **metrics,
                })
    return pd.DataFrame(rows)


def summarize_robustness_curves(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    keys = ["dataset", "model", "method", "seed", "eta", "n"]
    clean = results[results["corruption"] == "clean"][keys + ["macro_f1", "accuracy"]].copy()
    clean = clean.rename(columns={"macro_f1": "clean_macro_f1", "accuracy": "clean_accuracy"})
    rows = []
    for group_keys, group in results[results["corruption"] != "clean"].groupby(keys + ["corruption"], dropna=False):
        metadata = dict(zip(keys + ["corruption"], group_keys))
        clean_match = clean.copy()
        for key in keys:
            value = metadata[key]
            clean_match = clean_match[clean_match[key].fillna("__NA__") == ("__NA__" if pd.isna(value) else value)]
        if clean_match.empty:
            continue
        clean_f1 = float(clean_match.iloc[0]["clean_macro_f1"])
        ordered = group.sort_values("severity")
        x = np.concatenate([[0.0], ordered["severity"].to_numpy(dtype=float)])
        y = np.concatenate([[clean_f1], ordered["macro_f1"].to_numpy(dtype=float)])
        relative = y / max(clean_f1, 1e-12)
        rows.append({
            **metadata,
            "clean_macro_f1": clean_f1,
            "relative_auc": float(getattr(np, "trapezoid", np.trapz)(relative, x) / max(x.max() - x.min(), 1.0)),
            "degradation_slope": float(np.polyfit(x, y, 1)[0]),
            "worst_severity_macro_f1": float(y[-1]),
            "mean_relative_drop": float(np.mean(1.0 - relative[1:])),
        })
    return pd.DataFrame(rows)

def run_prediction_error_suite(
    results_df: pd.DataFrame,
    max_batches: int = 5,
) -> pd.DataFrame:
    frames = []
    ok = results_df[
        (results_df["status"] == "ok")
        & (results_df["method"].isin(["FixedPred", "Strict"]))
    ].copy()
    for _, row in ok.iterrows():
        model, _ = load_checkpoint(
            MODEL_FACTORIES[row["model"]],
            Path(row["checkpoint"]),
        )
        test_loader = build_dataloaders(
            row["dataset"],
            split_seed=CFG.split_seed,
            loader_seed=CFG.loader_seed,
        )["test"]
        diagnostic_plan = {"clean": [1]}
        diagnostic_plan.update({
            name: list(CFG.diagnostic_severities)
            for name in CFG.diagnostic_corruptions
        })
        for corruption, severities in diagnostic_plan.items():
            for severity in severities:
                table = prediction_error_diagnostics(
                    model,
                    test_loader,
                    dataset_name=row["dataset"],
                    method=row["method"],
                    eta=float(row["eta"]),
                    n=int(row["n"]),
                    corruption=corruption,
                    severity=int(severity),
                    max_batches=max_batches,
                )
                if table.empty:
                    continue
                table.insert(0, "dataset", row["dataset"])
                table.insert(1, "model", row["model"])
                table.insert(2, "seed", int(row["seed"]))
                table.insert(3, "split_seed", CFG.split_seed)
                frames.append(table)
    return (
        pd.concat(frames, ignore_index=True)
        if frames
        else pd.DataFrame()
    )


def run_timing_on_checkpoints(
    results_df: pd.DataFrame,
) -> pd.DataFrame:
    frames = []
    ok = results_df[results_df["status"] == "ok"].copy()
    for _, row in ok.iterrows():
        model, _ = load_checkpoint(
            MODEL_FACTORIES[row["model"]],
            Path(row["checkpoint"]),
        )
        for timing_batch_size in CFG.timing_batch_sizes:
            validation_loader = build_dataloaders(
                row["dataset"],
                split_seed=CFG.split_seed,
                loader_seed=CFG.loader_seed,
                batch_size=int(timing_batch_size),
            )["validation"]
            xb, yb = next(iter(validation_loader))
            table = timing_suite(
                model,
                xb,
                yb,
                method=row["method"],
                eta=(None if pd.isna(row.get("eta")) else float(row["eta"])),
                n=(None if pd.isna(row.get("n")) else int(row["n"])),
            )
            table.insert(0, "dataset", row["dataset"])
            table.insert(1, "model", row["model"])
            table.insert(2, "trained_method", row["method"])
            table.insert(3, "seed", int(row["seed"]))
            frames.append(table)
    return (
        pd.concat(frames, ignore_index=True)
        if frames
        else pd.DataFrame()
    )

robustness_results = pd.DataFrame()
prediction_error_results = pd.DataFrame()
timing_results = pd.DataFrame()
robustness_summary_results = pd.DataFrame()
if RUN_ROBUSTNESS_ANALYSIS:
    if resolved_final_results.empty:
        raise FileNotFoundError("Final results are required")
    robustness_results = run_robustness_suite(
        resolved_final_results
    )
    prediction_error_results = run_prediction_error_suite(
        resolved_final_results
    )
    timing_results = run_timing_on_checkpoints(
        resolved_final_results
    )
    robustness_summary_results = summarize_robustness_curves(robustness_results)
    robustness_results.to_csv(
        METRIC_DIR / "robustness_results.csv",
        index=False,
    )
    robustness_summary_results.to_csv(
        METRIC_DIR / "robustness_curve_summary.csv",
        index=False,
    )
    prediction_error_results.to_csv(
        METRIC_DIR / "prediction_error_per_sample_layer.csv",
        index=False,
    )
    timing_results.to_csv(
        METRIC_DIR / "timing_results.csv",
        index=False,
    )
    print("robustness", robustness_results.shape)
    print("robustness summary", robustness_summary_results.shape)
    print("prediction errors", prediction_error_results.shape)
    print("timing", timing_results.shape)

## 11B. Вычислительно сопоставимое сравнение

Этот блок повторно обучает выбранные методы в пределах одного wall-clock бюджета, определенного по BP для того же dataset и seed. Test не используется для определения бюджета. Блок является обязательным для публикационного пакета и опциональным для базовой ВКР.

In [ ]:
# 18C. Equal-wall-clock, optimizer и split ablations

def train_budget_pass(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    method: str,
    eta: Optional[float],
    n: Optional[int],
    remaining_sec: float,
) -> dict[str, float]:
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_samples = 0
    completed_batches = 0
    start = time.perf_counter()
    for xb, yb in loader:
        if completed_batches > 0 and (time.perf_counter() - start) >= remaining_sec:
            break
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        if method == "BP":
            logits = model(xb)
            ensure_finite_tensor("budget_logits", logits)
            loss = loss_fn(logits, yb)
            ensure_finite_tensor("budget_loss", loss)
            loss.backward()
        else:
            _, loss, _, _, _ = PCInfer(
                model, loss_fn, xb, yb, method,
                eta=0.1 if eta is None else float(eta),
                n=20 if n is None else int(n),
            )
            ensure_finite_tensor("budget_pc_loss", loss)
        gradient_diagnostics(model)
        optimizer.step()
        ensure_finite_parameters(model)
        batch_size = int(yb.shape[0])
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size
        completed_batches += 1
    sync_device()
    elapsed = time.perf_counter() - start
    if total_samples == 0:
        return {"train_loss": float("nan"), "elapsed_sec": elapsed, "train_samples": 0, "completed_batches": 0}
    return {
        "train_loss": total_loss / total_samples,
        "elapsed_sec": elapsed,
        "train_samples": total_samples,
        "completed_batches": completed_batches,
    }


def fit_with_wall_clock_budget(
    model_factory: Callable[[], nn.Sequential],
    train_loader: DataLoader,
    validation_loader: DataLoader,
    method: str,
    seed: int,
    budget_sec: float,
    eta: Optional[float],
    n: Optional[int],
    optimizer_name: str,
    learning_rate: float,
    checkpoint_path: Path,
    max_epochs: int = 100,
) -> tuple[nn.Module, pd.DataFrame, dict[str, Any]]:
    set_seed(seed)
    model = model_factory().to(DEVICE)
    optimizer = make_optimizer(model, optimizer_name, learning_rate)
    best_metric = -np.inf
    best_state = None
    best_epoch = -1
    rows = []
    elapsed_total = 0.0
    for epoch in range(1, max_epochs + 1):
        if elapsed_total >= budget_sec and epoch > 1:
            break
        remaining_sec = max(float(budget_sec) - elapsed_total, 0.0)
        info = train_budget_pass(
            model, train_loader, optimizer,
            method=method, eta=eta, n=n,
            remaining_sec=remaining_sec,
        )
        if int(info["train_samples"]) == 0:
            break
        elapsed_total += float(info["elapsed_sec"])
        val = evaluate_classifier(model, validation_loader)
        rows.append({
            "epoch": epoch,
            "cumulative_train_time_sec": elapsed_total,
            "epoch_time_sec": float(info["elapsed_sec"]),
            "train_loss": float(info["train_loss"]),
            "train_samples": int(info["train_samples"]),
            "completed_batches": int(info["completed_batches"]),
            "val_loss": val["loss"],
            "val_accuracy": val["accuracy"],
            "val_macro_f1": val["macro_f1"],
        })
        score = float(val[CFG.primary_metric])
        if score > best_metric + 1e-12:
            best_metric = score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
    if best_state is None:
        raise RuntimeError("No valid checkpoint within wall-clock budget")
    model.load_state_dict(best_state)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    metadata = {
        "method": method,
        "seed": seed,
        "budget_sec": budget_sec,
        "elapsed_sec": elapsed_total,
        "best_epoch": best_epoch,
        "best_validation_metric": best_metric,
        "optimizer": optimizer_name,
        "eta": eta,
        "n": n,
    }
    torch.save({"state_dict": best_state, "metadata": metadata}, checkpoint_path)
    return model, pd.DataFrame(rows), metadata


def run_equal_wall_clock_suite(results_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    ok = results_df[results_df["status"] == "ok"].copy()
    ok = ok[ok["seed"].isin(CFG.compute_matched_seeds)]
    for (dataset_name, model_name, seed), group in ok.groupby(["dataset", "model", "seed"]):
        bp = group[group["method"] == "BP"]
        if bp.empty:
            continue
        budget_sec = float(bp.iloc[0]["total_train_time_sec"])
        for _, method_row in group.iterrows():
            method = method_row["method"]
            optimizer_name = str(method_row.get("optimizer", CFG.optimizer_name))
            learning_rate = float(method_row.get("learning_rate", CFG.learning_rate))
            loaders_local = build_dataloaders(
                dataset_name,
                split_seed=int(method_row.get("split_seed", CFG.split_seed)),
                loader_seed=int(method_row.get("loader_seed", CFG.loader_seed)),
            )
            run_id = f"wallclock_{dataset_name}_{model_name}_{method}_seed{seed}"
            checkpoint = CHECKPOINT_DIR / f"{run_id}.pt"
            model, history, metadata = fit_with_wall_clock_budget(
                MODEL_FACTORIES[model_name],
                loaders_local["train"], loaders_local["validation"],
                method=method, seed=int(seed), budget_sec=budget_sec,
                eta=None if pd.isna(method_row.get("eta")) else float(method_row["eta"]),
                n=None if pd.isna(method_row.get("n")) else int(method_row["n"]),
                optimizer_name=optimizer_name,
                learning_rate=learning_rate,
                checkpoint_path=checkpoint,
            )
            test_metrics = evaluate_classifier(model, loaders_local["test"])
            history.to_csv(METRIC_DIR / f"{run_id}_history.csv", index=False)
            rows.append({
                "dataset": dataset_name,
                "model": model_name,
                "method": method,
                "seed": int(seed),
                "budget_sec": budget_sec,
                "actual_train_time_sec": metadata["elapsed_sec"],
                "best_epoch": metadata["best_epoch"],
                "test_loss": test_metrics["loss"],
                "test_accuracy": test_metrics["accuracy"],
                "test_macro_f1": test_metrics["macro_f1"],
                "checkpoint": str(checkpoint),
            })
    return pd.DataFrame(rows)


def run_optimizer_ablation_suite(selected: dict[str, Any]) -> pd.DataFrame:
    rows = []
    dataset_name = "FashionMNIST"
    for optimizer_name in CFG.optimizer_ablation_names:
        lr = CFG.sgd_learning_rate if optimizer_name.lower() == "sgd" else CFG.learning_rate
        for seed in CFG.optimizer_ablation_seeds:
            for method in ["BP", "FixedPred", "Strict"]:
                params = selected[dataset_name].get(method, {}) if method != "BP" else {}
                row, _ = final_run(
                    dataset_name, "lenet_classic", method, seed,
                    params.get("eta"), params.get("n"), CFG.final_epochs,
                    optimizer_name=optimizer_name,
                    learning_rate=lr,
                    run_prefix="optimizer_ablation",
                )
                rows.append(row)
    return pd.DataFrame(rows)


def run_split_sensitivity_suite(selected: dict[str, Any]) -> pd.DataFrame:
    rows = []
    dataset_name = "FashionMNIST"
    for split_seed in CFG.split_sensitivity_seeds:
        for model_seed in CFG.split_sensitivity_model_seeds:
            for method in ["BP", "FixedPred", "Strict"]:
                params = selected[dataset_name].get(method, {}) if method != "BP" else {}
                row, _ = final_run(
                    dataset_name, "lenet_classic", method, model_seed,
                    params.get("eta"), params.get("n"), CFG.final_epochs,
                    split_seed_value=int(split_seed),
                    loader_seed_value=CFG.loader_seed,
                    run_prefix=f"split_sensitivity_split{split_seed}",
                )
                rows.append(row)
    return pd.DataFrame(rows)

compute_matched_results = pd.DataFrame()
optimizer_ablation_results = pd.DataFrame()
split_sensitivity_results = pd.DataFrame()
if RUN_COMPUTE_MATCHED_ANALYSIS:
    compute_matched_results = run_equal_wall_clock_suite(resolved_final_results)
    compute_matched_results.to_csv(METRIC_DIR / "compute_matched_results.csv", index=False)
if RUN_OPTIMIZER_ABLATION:
    selected_all = json.loads(SELECTED_CONFIG_PATH.read_text(encoding="utf-8"))
    optimizer_ablation_results = run_optimizer_ablation_suite(selected_all)
    optimizer_ablation_results.to_csv(METRIC_DIR / "optimizer_ablation_results.csv", index=False)
if RUN_SPLIT_SENSITIVITY:
    selected_all = json.loads(SELECTED_CONFIG_PATH.read_text(encoding="utf-8"))
    split_sensitivity_results = run_split_sensitivity_suite(selected_all)
    split_sensitivity_results.to_csv(METRIC_DIR / "split_sensitivity_results.csv", index=False)


## 12. Динамика PCInfer и устойчивость результатов

Раздел реализует три ограниченных диагностических вопроса из плана. Он использует только выбранные и обученные checkpoints и не участвует в выборе гиперпараметров.

- D1: сходится ли обучающий сигнал при росте `n`;
- D2: устойчивы ли результаты между seed и сколько запусков завершается неудачей;
- D3: какие слои сильнее реагируют на шум, размытие и перекрытие.

In [ ]:
# 19. Траектория PCInfer, stability summary и послойные тренды искажений

def _state_map(values: Any, batch_size: int) -> dict[str, torch.Tensor]:
    result: dict[str, torch.Tensor] = {}
    if values is None:
        return result
    for layer_idx, value in enumerate(values):
        if not torch.is_tensor(value):
            continue
        tensor = value.detach()
        if tensor.ndim == 0 or tensor.shape[0] != batch_size:
            continue
        result[str(layer_idx)] = tensor.float().reshape(batch_size, -1).cpu()
    return result


def pc_budget_snapshot(
    model: nn.Module,
    xb: torch.Tensor,
    yb: torch.Tensor,
    method: str,
    eta: float,
    n: int,
) -> dict[str, Any]:
    work = copy.deepcopy(model).to(DEVICE)
    work.eval()
    work.zero_grad(set_to_none=True)
    output = PCInfer(
        work, nn.CrossEntropyLoss(), xb, yb, method, eta=float(eta), n=int(n)
    )
    beliefs, loss, epsilon = output[0], output[1], output[-1]
    error_rows = []
    for layer_idx, value in enumerate(epsilon):
        if not torch.is_tensor(value):
            continue
        error_rows.append({
            "layer": layer_idx,
            "epsilon_rms": float(torch.sqrt(torch.mean(value.detach().float() ** 2))),
        })
    return {
        "n": int(n),
        "loss": float(loss.item()),
        "gradients": named_gradient_vectors(work),
        "beliefs": _state_map(beliefs, int(yb.shape[0])),
        "errors": pd.DataFrame(error_rows),
    }


def inference_trajectory_diagnostics(
    model: nn.Module,
    loader: DataLoader,
    method: str,
    eta: float,
    n_values: list[int],
) -> dict[str, pd.DataFrame]:
    xb, yb = next(iter(loader))
    xb = xb[: CFG.trajectory_max_samples].to(DEVICE)
    yb = yb[: CFG.trajectory_max_samples].to(DEVICE)
    budgets = sorted({int(n) for n in n_values if int(n) >= 1})
    if not budgets:
        raise ValueError("n_values must contain at least one positive integer")

    bp_gradients, bp_loss = gradient_for_method(
        copy.deepcopy(model).to(DEVICE), xb, yb, "BP"
    )
    snapshots = {
        n: pc_budget_snapshot(model, xb, yb, method, eta, n)
        for n in budgets
    }
    terminal = snapshots[max(budgets)]

    gradient_rows, belief_rows, error_rows = [], [], []
    for n, snapshot in snapshots.items():
        comparison = compare_gradient_maps(bp_gradients, snapshot["gradients"])
        gradient_rows.append({
            "n": n,
            "method": method,
            "eta": eta,
            "bp_loss": bp_loss,
            "pc_loss": snapshot["loss"],
            "mean_cosine_to_bp": float(comparison["cosine"].mean()),
            "min_cosine_to_bp": float(comparison["cosine"].min()),
            "mean_relative_l2_to_bp": float(comparison["relative_l2"].mean()),
            "max_relative_l2_to_bp": float(comparison["relative_l2"].max()),
        })
        for layer, state in snapshot["beliefs"].items():
            terminal_state = terminal["beliefs"].get(layer)
            if terminal_state is None or state.shape[0] < 2:
                continue
            belief_rows.append({
                "n": n,
                "method": method,
                "eta": eta,
                "layer": layer,
                "cka_to_terminal_budget": linear_cka(state, terminal_state),
                "terminal_n": max(budgets),
            })
        if not snapshot["errors"].empty:
            local = snapshot["errors"].copy()
            local.insert(0, "n", n)
            local.insert(1, "method", method)
            local.insert(2, "eta", eta)
            error_rows.append(local)

    return {
        "gradients": pd.DataFrame(gradient_rows),
        "beliefs": pd.DataFrame(belief_rows),
        "errors": pd.concat(error_rows, ignore_index=True) if error_rows else pd.DataFrame(),
    }


def run_inference_trajectory_suite(results_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    sinks = {"gradients": [], "beliefs": [], "errors": []}
    ok = results_df[
        (results_df["status"] == "ok")
        & (results_df["method"].isin(["FixedPred", "Strict"]))
    ].copy()
    for _, row in ok.iterrows():
        model, _ = load_checkpoint(MODEL_FACTORIES[row["model"]], Path(row["checkpoint"]))
        validation_loader = build_dataloaders(
            row["dataset"],
            split_seed=CFG.split_seed,
            loader_seed=CFG.loader_seed,
        )["validation"]
        n_values = sorted(set(CFG.trajectory_n_values + [int(row["n"]), len(model)]))
        tables = inference_trajectory_diagnostics(
            model, validation_loader, row["method"], float(row["eta"]), n_values
        )
        common = {
            "dataset": row["dataset"],
            "model": row["model"],
            "trained_method": row["method"],
            "seed": int(row["seed"]),
            "trained_eta": float(row["eta"]),
            "trained_n": int(row["n"]),
        }
        for name, table in tables.items():
            if table.empty:
                continue
            enriched = table.copy()
            for key, value in reversed(list(common.items())):
                enriched.insert(0, key, value)
            sinks[name].append(enriched)
    return {
        name: pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        for name, frames in sinks.items()
    }


def stability_summary(
    results_df: pd.DataFrame, metric: str = "test_macro_f1"
) -> pd.DataFrame:
    rows = []
    group_cols = ["dataset", "model", "method", "eta", "n"]
    for keys, group in results_df.groupby(group_cols, dropna=False):
        success = group[group["status"] == "ok"]
        values = success[metric].dropna().astype(float).to_numpy() if metric in success else np.array([])
        mean = float(np.mean(values)) if len(values) else np.nan
        std = float(np.std(values, ddof=1)) if len(values) > 1 else np.nan
        rows.append({
            **dict(zip(group_cols, keys)),
            "n_attempted": int(len(group)),
            "n_success": int(len(success)),
            "success_rate": float(len(success) / len(group)) if len(group) else np.nan,
            "metric": metric,
            "mean": mean,
            "std": std,
            "coefficient_of_variation": (
                float(std / abs(mean))
                if np.isfinite(std) and np.isfinite(mean) and mean != 0
                else np.nan
            ),
        })
    return pd.DataFrame(rows)


def seed_rank_stability(
    results_df: pd.DataFrame, metric: str = "test_macro_f1"
) -> pd.DataFrame:
    rows = []
    ok = results_df[results_df["status"] == "ok"].copy()
    for (dataset_name, model_name), group in ok.groupby(["dataset", "model"]):
        pivot = group.pivot_table(index="method", columns="seed", values=metric, aggfunc="mean")
        seeds = list(pivot.columns)
        for i, seed_a in enumerate(seeds):
            for seed_b in seeds[i + 1:]:
                pair = pivot[[seed_a, seed_b]].dropna()
                if len(pair) < 3:
                    continue
                result = stats.spearmanr(pair[seed_a], pair[seed_b])
                rows.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "seed_a": int(seed_a),
                    "seed_b": int(seed_b),
                    "n_methods": int(len(pair)),
                    "spearman_rank": float(result.statistic),
                    "p_value": float(result.pvalue),
                    "analysis_mode": "descriptive_rank_stability",
                })
    return pd.DataFrame(rows)


def failure_catalog(results_df: pd.DataFrame) -> pd.DataFrame:
    failed = results_df[results_df["status"] != "ok"].copy()
    if failed.empty:
        return pd.DataFrame(columns=["dataset", "model", "method", "error", "count"])
    failed["error"] = failed.get("error", "unspecified").fillna("unspecified")
    return (
        failed.groupby(["dataset", "model", "method", "error"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )


def prediction_error_layer_trends(table: pd.DataFrame) -> pd.DataFrame:
    if table.empty:
        return pd.DataFrame()
    local = table[table["corruption"] != "clean"].copy()
    rows = []
    group_cols = ["dataset", "model", "method", "seed", "corruption", "layer"]
    for keys, group in local.groupby(group_cols, dropna=False):
        severity_mean = group.groupby("severity", as_index=False)["epsilon_rms"].mean()
        if severity_mean["severity"].nunique() < 2:
            continue
        result = stats.spearmanr(severity_mean["severity"], severity_mean["epsilon_rms"])
        base = float(severity_mean.sort_values("severity")["epsilon_rms"].iloc[0])
        last = float(severity_mean.sort_values("severity")["epsilon_rms"].iloc[-1])
        rows.append({
            **dict(zip(group_cols, keys)),
            "spearman_severity_epsilon": float(result.statistic),
            "p_value": float(result.pvalue),
            "relative_growth_first_to_last": (last - base) / (abs(base) + 1e-12),
        })
    return pd.DataFrame(rows)

trajectory_tables: dict[str, pd.DataFrame] = {}
stability_results = pd.DataFrame()
rank_stability_results = pd.DataFrame()
failure_results = pd.DataFrame()
layer_error_trends = pd.DataFrame()

if RUN_INFERENCE_TRAJECTORY:
    if resolved_final_results.empty:
        raise FileNotFoundError("Final results are required for inference trajectory")
    trajectory_tables = run_inference_trajectory_suite(resolved_final_results)
    for name, table in trajectory_tables.items():
        table.to_csv(METRIC_DIR / f"pcinfer_trajectory_{name}.csv", index=False)
        print("trajectory", name, table.shape)

if RUN_STABILITY_ANALYSIS:
    if resolved_final_results.empty:
        raise FileNotFoundError("Final results are required for stability analysis")
    stability_results = stability_summary(resolved_final_results)
    rank_stability_results = seed_rank_stability(resolved_final_results)
    failure_results = failure_catalog(resolved_final_results)
    stability_results.to_csv(METRIC_DIR / "stability_summary.csv", index=False)
    rank_stability_results.to_csv(METRIC_DIR / "seed_rank_stability.csv", index=False)
    failure_results.to_csv(METRIC_DIR / "failure_catalog.csv", index=False)

resolved_prediction_errors = load_results_table(
    prediction_error_results, METRIC_DIR / "prediction_error_per_sample_layer.csv"
)
if not resolved_prediction_errors.empty:
    layer_error_trends = prediction_error_layer_trends(resolved_prediction_errors)
    layer_error_trends.to_csv(METRIC_DIR / "prediction_error_layer_trends.csv", index=False)
    print("layer error trends", layer_error_trends.shape)

## 13. Статистический анализ

In [ ]:
# 19. Агрегация, доверительные интервалы и парные сравнения

def summarize_with_ci(
    df: pd.DataFrame,
    group_cols: list[str],
    metric_cols: list[str],
    confidence: float = 0.95,
) -> pd.DataFrame:
    rows = []
    for keys, group in df.groupby(group_cols, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        base = dict(zip(group_cols, keys))
        for metric in metric_cols:
            values = group[metric].dropna().astype(float).to_numpy()
            n = len(values)
            mean = float(np.mean(values)) if n else np.nan
            std = float(np.std(values, ddof=1)) if n > 1 else np.nan
            if n > 1:
                critical = stats.t.ppf((1 + confidence) / 2, df=n - 1)
                half = float(critical * std / math.sqrt(n))
            else:
                half = np.nan
            rows.append({
                **base,
                "metric": metric,
                "n": n,
                "mean": mean,
                "std": std,
                "ci_low": mean - half if np.isfinite(half) else np.nan,
                "ci_high": mean + half if np.isfinite(half) else np.nan,
            })
    return pd.DataFrame(rows)


def cohen_dz(differences: np.ndarray) -> float:
    if len(differences) < 2:
        return np.nan
    std = np.std(differences, ddof=1)
    if std == 0:
        return np.nan
    return float(np.mean(differences) / std)


def holm_adjust(p_values: list[float]) -> list[float]:
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted.tolist()


def inference_label(n_pairs: int) -> str:
    if n_pairs >= CFG.confirmatory_min_pairs:
        return "confirmatory_eligible"
    if n_pairs >= CFG.wilcoxon_min_pairs:
        return "exploratory_inference"
    return "descriptive_only"


def paired_method_comparisons(
    df: pd.DataFrame,
    metric: str = "test_macro_f1",
    reference_method: str = "BP",
) -> pd.DataFrame:
    rows = []
    methods = [
        method
        for method in sorted(df["method"].dropna().unique())
        if method != reference_method
    ]
    for dataset_name in sorted(df["dataset"].dropna().unique()):
        local = df[
            (df["dataset"] == dataset_name)
            & (df["status"] == "ok")
        ]
        reference = local[
            local["method"] == reference_method
        ][["seed", metric]].rename(columns={metric: "reference"})

        for method in methods:
            candidate = local[
                local["method"] == method
            ][["seed", metric]].rename(columns={metric: "candidate"})
            paired = reference.merge(candidate, on="seed", how="inner")
            differences = (
                paired["candidate"].to_numpy()
                - paired["reference"].to_numpy()
            )
            n_pairs = len(differences)
            analysis_mode = inference_label(n_pairs)
            test_performed = False
            test_reason = (
                f"Need at least {CFG.wilcoxon_min_pairs} paired seeds"
            )
            p_value = np.nan
            statistic = np.nan

            if (
                n_pairs >= CFG.wilcoxon_min_pairs
                and np.any(differences != 0)
            ):
                try:
                    test = stats.wilcoxon(differences)
                    p_value = float(test.pvalue)
                    statistic = float(test.statistic)
                    test_performed = True
                    test_reason = "Wilcoxon performed"
                except ValueError as exc:
                    test_reason = f"Wilcoxon skipped: {exc}"
            elif n_pairs >= CFG.wilcoxon_min_pairs:
                test_reason = "Wilcoxon skipped: all paired differences are zero"

            rows.append({
                "dataset": dataset_name,
                "reference": reference_method,
                "candidate": method,
                "metric": metric,
                "n_pairs": n_pairs,
                "minimum_pairs_required": CFG.wilcoxon_min_pairs,
                "confirmatory_pairs_required": CFG.confirmatory_min_pairs,
                "analysis_mode": analysis_mode,
                "test_performed": test_performed,
                "test_reason": test_reason,
                "mean_difference": (
                    float(np.mean(differences))
                    if n_pairs
                    else np.nan
                ),
                "cohen_dz": cohen_dz(differences),
                "wilcoxon_statistic": statistic,
                "p_value": p_value,
            })

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    valid = result["test_performed"] & result["p_value"].notna()
    result["p_value_holm"] = np.nan
    if valid.any():
        result.loc[valid, "p_value_holm"] = holm_adjust(
            result.loc[valid, "p_value"].tolist()
        )
    return result


def h4_tuned_and_transfer_gaps(
    final_df: pd.DataFrame,
    transfer_df: pd.DataFrame,
) -> pd.DataFrame:
    target = CFG.h4_target_dataset
    final_ok = final_df[
        (final_df["status"] == "ok")
        & (final_df["dataset"] == target)
        & (final_df["model"] == "lenet_classic")
    ].copy()
    bp = final_ok[
        final_ok["method"] == "BP"
    ][["seed", "test_macro_f1"]].rename(
        columns={"test_macro_f1": "bp_macro_f1"}
    )

    frames = []
    tuned = final_ok[
        final_ok["method"].isin(["FixedPred", "Strict"])
    ].copy()
    if not tuned.empty:
        tuned = tuned.merge(bp, on="seed", how="inner")
        tuned["configuration_origin"] = target
        tuned["analysis"] = "H4_tuned_per_dataset"
        tuned["macro_f1_gap_to_bp"] = (
            tuned["test_macro_f1"] - tuned["bp_macro_f1"]
        )
        frames.append(tuned)

    if transfer_df is not None and not transfer_df.empty:
        transferred = transfer_df[
            (transfer_df["status"] == "ok")
            & (transfer_df["dataset"] == target)
        ].copy()
        if not transferred.empty:
            transferred = transferred.merge(bp, on="seed", how="inner")
            transferred["configuration_origin"] = CFG.h4_source_dataset
            transferred["analysis"] = "H4_fixed_configuration_transfer"
            transferred["macro_f1_gap_to_bp"] = (
                transferred["test_macro_f1"]
                - transferred["bp_macro_f1"]
            )
            frames.append(transferred)

    return (
        pd.concat(frames, ignore_index=True)
        if frames
        else pd.DataFrame()
    )

In [ ]:
# 19A. Формирование итоговых статистических таблиц

resolved_final_results = load_results_table(
    final_results,
    METRIC_DIR / "final_results_by_seed.csv",
)
resolved_dataset_transfer = load_results_table(
    dataset_transfer_results,
    METRIC_DIR / "h4_dataset_transfer_results.csv",
)

final_summary = pd.DataFrame()
paired_comparisons = pd.DataFrame()
h4_comparison = pd.DataFrame()

if not resolved_final_results.empty:
    ok_final = resolved_final_results[
        resolved_final_results["status"] == "ok"
    ].copy()

    final_summary = summarize_with_ci(
        ok_final,
        group_cols=["dataset", "model", "method"],
        metric_cols=[
            "test_macro_f1",
            "test_accuracy",
            "test_loss",
        ],
    )
    paired_comparisons = paired_method_comparisons(
        ok_final,
        metric="test_macro_f1",
    )
    h4_comparison = h4_tuned_and_transfer_gaps(
        resolved_final_results,
        resolved_dataset_transfer,
    )

    final_summary.to_csv(
        METRIC_DIR / "final_summary_ci95.csv",
        index=False,
    )
    paired_comparisons.to_csv(
        METRIC_DIR / "paired_method_comparisons.csv",
        index=False,
    )
    if not h4_comparison.empty:
        h4_comparison.to_csv(
            METRIC_DIR / "h4_tuned_and_transfer_gaps.csv",
            index=False,
        )

    display(final_summary)
    display(paired_comparisons)
    if not h4_comparison.empty:
        display(h4_comparison)
else:
    print("Final results are not available yet")

## 14. H9: переносимость конфигураций между вариантами LeNet

Полная поисковая сетка не повторяется для каждой архитектуры. Значения `eta` и `n`, выбранные на LeNet-classic, переносятся без дополнительной настройки на LeNet-modern и LeNet-small.

Этот эксперимент проверяет переносимость настроек, а не чистую чувствительность алгоритма к архитектуре. BP обучается отдельно для каждой архитектуры и используется как локальная базовая модель.

In [ ]:
# 20. H9: переносимость выбранных конфигураций между архитектурами

def run_architecture_transfer_suite(
    dataset_name: str,
    selected: dict[str, Any],
    seeds: list[int],
) -> pd.DataFrame:
    rows = []
    for model_name in MODEL_FACTORIES:
        for seed in seeds:
            methods = [
                ("BP", None, None),
                (
                    "FixedPred",
                    selected["FixedPred"]["eta"],
                    selected["FixedPred"]["n"],
                ),
                (
                    "Strict",
                    selected["Strict"]["eta"],
                    selected["Strict"]["n"],
                ),
            ]
            for method, eta, n in methods:
                try:
                    row, _ = final_run(
                        dataset_name,
                        model_name,
                        method,
                        seed,
                        eta,
                        n,
                        CFG.final_epochs,
                        selection_origin=dataset_name,
                    )
                    row["analysis"] = "H9_configuration_transferability"
                    row["selection_architecture"] = "lenet_classic"
                    row["evaluation_architecture"] = model_name
                except Exception as exc:
                    row = {
                        "analysis": "H9_configuration_transferability",
                        "dataset": dataset_name,
                        "model": model_name,
                        "evaluation_architecture": model_name,
                        "selection_architecture": "lenet_classic",
                        "method": method,
                        "seed": seed,
                        "model_seed": seed,
                        "split_seed": CFG.split_seed,
                        "loader_seed": CFG.loader_seed,
                        "eta": eta,
                        "n": n,
                        "status": "failed",
                        "error": repr(exc),
                    }
                rows.append(row)
    return pd.DataFrame(rows)


architecture_transfer_results = pd.DataFrame()
if RUN_ARCHITECTURE_TRANSFER_ANALYSIS:
    if not SELECTED_CONFIG_PATH.exists():
        raise FileNotFoundError("selected_configs.json is required")
    selected_all = json.loads(
        SELECTED_CONFIG_PATH.read_text(encoding="utf-8")
    )
    architecture_transfer_results = run_architecture_transfer_suite(
        CFG.datasets_required[0],
        selected_all[CFG.datasets_required[0]],
        CFG.final_seeds,
    )
    architecture_transfer_results.to_csv(
        METRIC_DIR / "architecture_transfer_results_by_seed.csv",
        index=False,
    )
    display(architecture_transfer_results)

## 15. Оценка вычислительного бюджета

In [ ]:
# 21. Оценка бюджета по пилотным измерениям

def estimate_runtime_hours(
    mean_epoch_seconds: float,
    epochs: int,
    n_runs: int,
    safety_factor: float = 1.25,
) -> float:
    return mean_epoch_seconds * epochs * n_runs * safety_factor / 3600.0

if not pilot_results.empty and "epoch_time_sec" in pilot_results.columns:
    successful = pilot_results[pilot_results["status"] == "ok"]
    mean_epoch = float(successful["epoch_time_sec"].mean())
    planned_runs = len(CFG.datasets_required) * len(CFG.final_seeds) * 3
    print("Estimated final training hours:", estimate_runtime_hours(mean_epoch, CFG.final_epochs, planned_runs))

## 16. Экспорт результатов

In [ ]:
# 22. Экспорт CSV/XLSX и манифеста

def csv_tables(directory: Path) -> dict[str, pd.DataFrame]:
    tables = {}
    for path in sorted(directory.glob("*.csv")):
        try:
            tables[path.stem[:31]] = pd.read_csv(path)
        except Exception as exc:
            print("Skipped", path, exc)
    return tables


def export_workbook(output_path: Path = RESULTS_DIR / "thesis_results_v11.xlsx") -> Path:
    tables = csv_tables(METRIC_DIR)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet, table in tables.items():
            table.to_excel(writer, sheet_name=sheet, index=False)
    return output_path


def results_manifest() -> dict[str, Any]:
    files = []
    for path in sorted(RESULTS_DIR.rglob("*")):
        if path.is_file():
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            files.append({"path": str(path), "sha256": digest, "bytes": path.stat().st_size})
    return {
        "created_utc": pd.Timestamp.utcnow().isoformat(),
        "torch2pc_commit": TORCH2PC_HEAD,
        "config": asdict(CFG),
        "files": files,
    }

# Выполнить после появления CSV:
# workbook_path = export_workbook()
# (RESULTS_DIR / "manifest.json").write_text(
#     json.dumps(results_manifest(), ensure_ascii=False, indent=2), encoding="utf-8"
# )
# print(workbook_path)

## 17. Финальный checklist

Перед использованием результатов в ВКР подтвердите:

- [ ] полный хеш Torch2PC зафиксирован;
- [ ] поправка Rosenbaum 2025 присутствует в литературе и source audit пройден;
- [ ] H0/H1 пройдены на CPU и на GPU при наличии;
- [ ] pilot выполнен минимум на трех seed;
- [ ] конфигурации выбраны по агрегированной validation-метрике;
- [ ] основное сравнение содержит 10 парных model_seed;
- [ ] test не использовался для выбора конфигураций;
- [ ] градиенты сравнивались минимум на пяти фиксированных пакетах;
- [ ] стратифицированная подвыборка CKA/RSA сохранена;
- [ ] bootstrap-интервалы CKA/RSA рассчитаны для публикационного пакета;
- [ ] equal-wall-clock сравнение завершено;
- [ ] рассчитаны AUC, наклон и worst-severity показатели устойчивости;
- [ ] Adam/SGD и split-sensitivity абляции выполнены или явно вынесены за границы ВКР;
- [ ] все NaN/Inf и исключения присутствуют в failure catalog;
- [ ] сохранены CSV/XLSX, рисунки, config, lock-файл и манифест;
- [ ] выполнен патентный поиск;
- [ ] положения, выносимые на защиту, содержат только подтвержденные числа;
- [ ] выводы отделяют подтвержденные результаты от предположений.

In [ ]:
# 23. Автоматический экспорт контейнерного запуска
if os.environ.get("AUTO_EXPORT_RESULTS", "1") == "1":
    csv_files = list(METRIC_DIR.glob("*.csv"))
    if csv_files:
        workbook_path = export_workbook(RESULTS_DIR / "thesis_results_v11.xlsx")
        print("Workbook:", workbook_path)
    manifest_path = RESULTS_DIR / "manifest_v11.json"
    manifest_path.write_text(
        json.dumps(results_manifest(), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Manifest:", manifest_path)